**Reviewer note:** This notebook is a preserved development experiment. Read `docs/EXPERIMENTS.md` and `docs/METHODOLOGY.md` before execution. Raw/restricted inputs are not included. Historical references to 'test' do not establish prospective independence or validated online causality.

# SRT - Calibrazione REAL/FEM e modello multi-fidelity

Notebook dedicato al confronto appaiato dei 868 casi REAL/FEM e ai modelli di discrepanza.

Principi:
- usa solo il dataset originale affidabile, senza i nuovi outlier;
- separa esplicitamente errore misurato e correzione da applicare;
- usa split cronologici e selezione esclusivamente sulla validation;
- confronta FEM grezzo, calibrazioni semplici, modello termico, modello residuale e surrogato FEM;
- il test finale non viene usato per scegliere modello o iperparametri.


In [ ]:
from pathlib import Path
import re
import math
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from IPython.display import display
except Exception:
    def display(x):
        print(x)

from sklearn.base import clone
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import RidgeCV, HuberRegressor, ElasticNetCV
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.decomposition import PCA

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

SEQ_LEN = 60                 # 60 two-minute samples span approximately two hours.
EXPECTED_STEP_SEC = 120      
MIN_STEP_SEC = 60             
MAX_GAP_SEC = 180            
MAX_SENSOR_RATE_C = 5.0       
USE_RESAMPLED_WINDOWS = True  
TARGET = "XEl Offset"        
SPEC_ARCSEC = 5.0             # Historical reference threshold; receiver-specific requirements differ.
INCLUDE_CURRENT_TEMP = True  


PROJECT_SEARCH_ROOTS = [Path.cwd(), *Path.cwd().parents, Path("/content/metrologyml"), Path.home() / "Downloads"]
DATA_SUBDIRS = [Path("."), Path("data/raw/real"), Path("data/raw/real/optional"), Path("data/raw/control"), Path("data/raw/fem"), Path("data/diagnostics/control"), Path("data/diagnostics/fem"), Path("data/diagnostics/history")]
DATA_DIRS = list(dict.fromkeys(root / subdir for root in PROJECT_SEARCH_ROOTS for subdir in DATA_SUBDIRS))
REAL_CANDIDATES = ["EffectiveDataset_Tseries_with_azimuth.csv", "EffectiveDataset_Tseries_total.csv", "EffectiveDataset_Tseries (2).xlsx", "EffectiveDataset_Tseries.xlsx", "EffectiveDataset_Tseries.csv"]
FEM_CANDIDATES = ["Dataset (1).xlsx", "dataset_multistep_jul_aug.csv", "Dataset.xlsx"]
POINTING_CANDIDATES = ["pointing_offsets_real_azimuth.csv"]
USE_POINTING_TARGETS_WITH_AZIMUTH = False  
MAX_ABS_XEL_FOR_TRAIN = 60.0    
MAX_ABS_EL_FOR_TRAIN = 180.0    
FILTER_MAINTENANCE_BY_NAME = False  # Maintenance scans are not rejected solely on their project label.
POINTING_CLEANING_MODE = "extreme_only"  
STABLE_JUMP_WINDOW_MIN = 180
STABLE_DXEL_JUMP_ARCSEC = 15.0
STABLE_DEL_JUMP_ARCSEC = 20.0

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 160)


In [ ]:
def find_data_file(candidates, data_dirs=DATA_DIRS):
    for d in data_dirs:
        for name in candidates:
            p = d / name
            if p.exists():
                return p
    raise FileNotFoundError(f"No file found among: {candidates}")


def read_table(path: Path) -> pd.DataFrame:
    """Read CSV/XLSX, with a direct openpyxl fallback for version conflicts."""
    path = Path(path)
    if path.suffix.lower() == ".csv":
        return pd.read_csv(path)
    if path.suffix.lower() in [".xlsx", ".xlsm", ".xls"]:
        try:
            return pd.read_excel(path, engine="openpyxl")
        except ImportError:
            from openpyxl import load_workbook
            wb = load_workbook(path, read_only=True, data_only=True)
            ws = wb.active
            rows = ws.iter_rows(values_only=True)
            header = list(next(rows))
            data = list(rows)
            wb.close()
            return pd.DataFrame(data, columns=header)
    raise ValueError(f"Unsupported format: {path}")


def normalize_sensor_columns(df: pd.DataFrame) -> pd.DataFrame:
    """Normalize sensor-column names."""
    mapping = {}
    for col in df.columns:
        m = re.match(r"^Sensors_\s*(\d+)$", str(col))
        if m:
            mapping[col] = f"Sensors_{int(m.group(1)):03d}"
    return df.rename(columns=mapping)


def get_sensor_cols(df: pd.DataFrame):
    cols = [c for c in df.columns if re.match(r"^Sensors_\d{3}$", str(c))]
    return sorted(cols, key=lambda c: int(c.split("_")[1]))


def ensure_datetime(df, col):
    df = df.copy()
    df[col] = pd.to_datetime(df[col], errors="coerce")
    return df


def xel_from_fem(df):
    """Project Accuracy_1 to XEL using cos(Elevation)."""
    out = df.copy()
    out["XEL"] = out["Accuracy_1"] * np.cos(np.deg2rad(out["Elevation"]))
    return out


In [ ]:
real_path = find_data_file(REAL_CANDIDATES)
print('Real dataset:', real_path)

real_raw = read_table(real_path)
real_raw = normalize_sensor_columns(real_raw)
real_raw = ensure_datetime(real_raw, "Timestamp")
real_raw = real_raw.sort_values("Timestamp").reset_index(drop=True)

sensor_cols = get_sensor_cols(real_raw)
print(f"Real rows: {len(real_raw):,} | Sensors: {len(sensor_cols)}")
print('Time range:', real_raw["Timestamp"].min(), "->", real_raw["Timestamp"].max())
print('Valid XEL targets:', real_raw["XEl Offset"].notna().sum(), "| EL:", real_raw["El Offset"].notna().sum())
print('Valid elevations:', real_raw["SRT Elevation Ange[DEG]"].notna().sum())
if "Azimuth" in real_raw.columns:
    print('Azimuth available:', real_raw["Azimuth"].notna().sum(), "| range:", real_raw["Azimuth"].min(), "->", real_raw["Azimuth"].max())
else:
    print('Azimuth is not present in the real dataset.')

real_raw[["SRT Elevation Ange[DEG]", "El Offset", "XEl Offset"]].describe()


In [ ]:
def audit_real_dataset(df, sensor_cols):
    print('Missing sensors:', int(df[sensor_cols].isna().sum().sum()))
    dt = df["Timestamp"].diff().dt.total_seconds().dropna()
    print('Temperature sampling interval [s]')
    print(dt.describe(percentiles=[0.5, 0.9, 0.95, 0.99]).to_string())

    target_df = df[df[TARGET].notna()].copy()
    target_dt = target_df["Timestamp"].diff().dt.total_seconds().dropna()
    print('\nValid cross-scan targets:', len(target_df))
    if len(target_dt):
        print('Time between targets [s]')
        print(target_dt.describe(percentiles=[0.5, 0.9, 0.95, 0.99]).to_string())

    flat = df[sensor_cols].to_numpy(dtype="float32").ravel()
    flat = flat[np.isfinite(flat)]
    print('\nReal temperatures: mean/std/min/max', np.mean(flat), np.std(flat), np.min(flat), np.max(flat))

audit_real_dataset(real_raw, sensor_cols)


In [ ]:
def _impute_sensor_frame(sensor_df, impute_limit=3):
    # Limited backward filling is offline-only and needs an online availability audit.
    return sensor_df.astype("float32").ffill(limit=impute_limit).bfill(limit=1)


def build_supervised_windows_raw(
    df,
    sensor_cols,
    target_col=TARGET,
    target_table=None,
    seq_len=SEQ_LEN,
    min_step_sec=MIN_STEP_SEC,
    max_gap_sec=MAX_GAP_SEC,
    max_sensor_rate_c=MAX_SENSOR_RATE_C,
    include_current=INCLUDE_CURRENT_TEMP,
    impute_limit=3,
):
    """Build consecutive-row windows, rejecting duplicate timestamps."""
    df = df.sort_values("Timestamp").reset_index(drop=True).copy()
    df[sensor_cols] = _impute_sensor_frame(df[sensor_cols], impute_limit=impute_limit)

    if target_table is not None:
        print('Azimuth in target_table is fully supported by resampled_grid; raw_rows uses targets from the thermal dataset.')
    target_mask = df[target_col].notna() & df["SRT Elevation Ange[DEG]"].notna()
    target_positions = np.flatnonzero(target_mask.to_numpy())

    X, elev, y, times = [], [], [], []
    skip = {"too_early": 0, "time_gap": 0, "too_close": 0, "nan": 0, "thermal_jump": 0}

    for pos in target_positions:
        end = pos + 1 if include_current else pos
        start = end - seq_len
        if start < 0:
            skip["too_early"] += 1
            continue

        window = df.iloc[start:end]
        dts = window["Timestamp"].diff().dt.total_seconds().dropna()
        if len(dts) != seq_len - 1 or (dts > max_gap_sec).any():
            skip["time_gap"] += 1
            continue
        if (dts < min_step_sec).any():
            skip["too_close"] += 1
            continue

        values = window[sensor_cols].to_numpy(dtype="float32")
        if not np.isfinite(values).all():
            skip["nan"] += 1
            continue
        if np.abs(np.diff(values, axis=0)).max() > max_sensor_rate_c:
            skip["thermal_jump"] += 1
            continue

        X.append(values)
        elev.append(float(df.loc[pos, "SRT Elevation Ange[DEG]"]))
        y.append(float(df.loc[pos, target_col]))
        times.append(df.loc[pos, "Timestamp"])

    meta = pd.DataFrame({"Timestamp": times, "Elevation": elev, "Target": y})
    return np.asarray(X, dtype="float32"), np.asarray(elev, dtype="float32").reshape(-1, 1), np.asarray(y, dtype="float32"), meta, skip


def build_supervised_windows_resampled(
    df,
    sensor_cols,
    target_col=TARGET,
    target_table=None,
    seq_len=SEQ_LEN,
    expected_step_sec=EXPECTED_STEP_SEC,
    max_gap_sec=MAX_GAP_SEC,
    max_sensor_rate_c=MAX_SENSOR_RATE_C,
    include_current=INCLUDE_CURRENT_TEMP,
    impute_limit=3,
):
    """Build two-minute windows, aggregating duplicate rows and checking gaps and thermal jumps."""
    df = df.sort_values("Timestamp").copy()
    if target_table is not None:
        target_df = target_table.rename(columns={"Elevation": "SRT Elevation Ange[DEG]"}).copy()
    else:
        target_df = df[df[target_col].notna() & df["SRT Elevation Ange[DEG]"].notna()].copy()


    sensor_ts = df[["Timestamp"] + sensor_cols].dropna(subset=["Timestamp"]).copy()
    sensor_ts["GridTime"] = sensor_ts["Timestamp"].dt.floor(f"{expected_step_sec}s")
    grid = sensor_ts.groupby("GridTime", sort=True)[sensor_cols].mean()
    grid = _impute_sensor_frame(grid, impute_limit=impute_limit)

    X, elev, azim, y, times, filenames = [], [], [], [], [], []
    skip = {"too_early": 0, "missing_grid": 0, "nan": 0, "thermal_jump": 0}
    step = pd.Timedelta(seconds=expected_step_sec)

    for _, row in target_df.iterrows():
        target_time = row["Timestamp"]
        end_time = target_time.floor(f"{expected_step_sec}s") if include_current else (target_time.floor(f"{expected_step_sec}s") - step)
        expected_index = pd.date_range(end=end_time, periods=seq_len, freq=step)

        if len(grid) == 0 or expected_index[0] < grid.index.min():
            skip["too_early"] += 1
            continue

        # Reject incomplete grids rather than filling long resampling gaps.
        if not expected_index.isin(grid.index).all():
            skip["missing_grid"] += 1
            continue

        values = grid.loc[expected_index, sensor_cols].to_numpy(dtype="float32")
        if not np.isfinite(values).all():
            skip["nan"] += 1
            continue
        if np.abs(np.diff(values, axis=0)).max() > max_sensor_rate_c:
            skip["thermal_jump"] += 1
            continue

        X.append(values)
        elev.append(float(row["SRT Elevation Ange[DEG]"]))
        azim.append(float(row["Azimuth"]) if "Azimuth" in row and pd.notna(row["Azimuth"]) else np.nan)
        y.append(float(row[target_col]))
        times.append(target_time)
        filenames.append(row["Filename"] if "Filename" in row else None)

    meta = pd.DataFrame({"Timestamp": times, "Elevation": elev, "Azimuth": azim, "Target": y, "Filename": filenames})
    return np.asarray(X, dtype="float32"), np.asarray(elev, dtype="float32").reshape(-1, 1), np.asarray(y, dtype="float32"), meta, skip


def build_supervised_windows(df, sensor_cols, **kwargs):
    if USE_RESAMPLED_WINDOWS:
        return build_supervised_windows_resampled(df, sensor_cols, **kwargs)
    return build_supervised_windows_raw(df, sensor_cols, **kwargs)

target_source = pointing_targets if USE_POINTING_TARGETS_WITH_AZIMUTH and "pointing_targets" in globals() and pointing_targets is not None else None
X_seq, X_elev, y, meta, skip = build_supervised_windows(real_raw, sensor_cols, target_table=target_source)
print('Window mode:', "resampled_grid" if USE_RESAMPLED_WINDOWS else "raw_rows")
print('Target source:', "pointing_offsets_real_azimuth" if target_source is not None else 'thermal_dataset')
print('Windows with azimuth:', int(meta["Azimuth"].notna().sum()) if "Azimuth" in meta else 0)
print('Valid supervised samples:', len(y))
print('Rejected by timing/jump checks:', skip)
print("Shape X_seq:", X_seq.shape, "X_elev:", X_elev.shape, "y:", y.shape)
meta.describe(include="all")


In [ ]:
def window_quality_report(X, meta):
    if len(X) == 0:
        print('No valid windows.')
        return pd.DataFrame()
    diffs = np.diff(X, axis=1)
    q = pd.DataFrame({
        "Timestamp": meta["Timestamp"].to_numpy(),
        "Elevation": meta["Elevation"].to_numpy(),
        "Target": meta["Target"].to_numpy(),
        "window_mean": X.mean(axis=(1, 2)),
        "window_std": X.std(axis=(1, 2)),
        "current_mean": X[:, -1, :].mean(axis=1),
        "trend_mean": (X[:, -1, :] - X[:, 0, :]).mean(axis=1),
        "abs_rate_mean": np.abs(diffs).mean(axis=(1, 2)),
        "abs_rate_max": np.abs(diffs).max(axis=(1, 2)),
    })
    print(q[["window_mean", "window_std", "current_mean", "trend_mean", "abs_rate_mean", "abs_rate_max"]].describe().to_string())
    suspect = q[(q["abs_rate_max"] > MAX_SENSOR_RATE_C) | (q["abs_rate_mean"] > 0.5)]
    print('Remaining suspect windows:', len(suspect))
    if len(suspect):
        display(suspect.sort_values("abs_rate_max", ascending=False).head(20))
    return q

quality_df = window_quality_report(X_seq, meta)


In [ ]:
def chronological_split(meta, train_frac=0.60, val_frac=0.20):
    order = np.argsort(meta["Timestamp"].to_numpy())
    n = len(order)
    n_train = int(n * train_frac)
    n_val = int(n * val_frac)
    train_idx = order[:n_train]
    val_idx = order[n_train:n_train + n_val]
    test_idx = order[n_train + n_val:]
    return train_idx, val_idx, test_idx

train_idx, val_idx, test_idx = chronological_split(meta)
print("Split sizes:", len(train_idx), len(val_idx), len(test_idx))
print("Train:", meta.iloc[train_idx]["Timestamp"].min(), "->", meta.iloc[train_idx]["Timestamp"].max())
print("Val  :", meta.iloc[val_idx]["Timestamp"].min(), "->", meta.iloc[val_idx]["Timestamp"].max())
print("Test :", meta.iloc[test_idx]["Timestamp"].min(), "->", meta.iloc[test_idx]["Timestamp"].max())

plt.figure(figsize=(10, 4))
plt.plot(meta["Timestamp"].to_numpy(), meta["Target"].to_numpy(), ".", alpha=0.75)
plt.axvspan(meta.iloc[train_idx]["Timestamp"].min(), meta.iloc[train_idx]["Timestamp"].max(), alpha=0.08, label="train")
plt.axvspan(meta.iloc[val_idx]["Timestamp"].min(), meta.iloc[val_idx]["Timestamp"].max(), alpha=0.08, label="val")
plt.axvspan(meta.iloc[test_idx]["Timestamp"].min(), meta.iloc[test_idx]["Timestamp"].max(), alpha=0.08, label="test")
plt.title(f"Observed target {TARGET} and chronological split")
plt.ylabel("arcsec")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()


In [ ]:
def scalar_physical_features(X, elev):
    current = X[:, -1, :]
    start = X[:, 0, :]
    mean_time = X.mean(axis=1)
    std_time = X.std(axis=1)
    diff = np.diff(X, axis=1)
    trend = current - start

    feats = pd.DataFrame({
        "elevation": elev.ravel(),
        "sin_el": np.sin(np.deg2rad(elev.ravel())),
        "cos_el": np.cos(np.deg2rad(elev.ravel())),
        "current_mean": current.mean(axis=1),
        "current_std_spatial": current.std(axis=1),
        "current_min": current.min(axis=1),
        "current_max": current.max(axis=1),
        "current_range": current.max(axis=1) - current.min(axis=1),
        "window_mean": X.mean(axis=(1, 2)),
        "window_std": X.std(axis=(1, 2)),
        "mean_sensor_std_time": std_time.mean(axis=1),
        "trend_mean": trend.mean(axis=1),
        "trend_std": trend.std(axis=1),
        "trend_range": trend.max(axis=1) - trend.min(axis=1),
        "abs_rate_mean": np.abs(diff).mean(axis=(1, 2)),
        "abs_rate_max": np.abs(diff).max(axis=(1, 2)),
    })
    return feats

def temporal_context_features(meta_df):
    """Build cyclic time features as seasonal and diurnal proxies."""
    ts = pd.to_datetime(meta_df["Timestamp"])
    hour = ts.dt.hour + ts.dt.minute / 60.0 + ts.dt.second / 3600.0
    doy = ts.dt.dayofyear.astype(float)
    out = pd.DataFrame({
        "hour_sin": np.sin(2 * np.pi * hour / 24.0),
        "hour_cos": np.cos(2 * np.pi * hour / 24.0),
        "doy_sin": np.sin(2 * np.pi * doy / 365.25),
        "doy_cos": np.cos(2 * np.pi * doy / 365.25),
        "month": ts.dt.month.astype(float),
    }).reset_index(drop=True)
    if "Azimuth" in meta_df.columns and meta_df["Azimuth"].notna().any():
        az = pd.to_numeric(meta_df["Azimuth"], errors="coerce").fillna(meta_df["Azimuth"].median())
        out["az_sin"] = np.sin(np.deg2rad(az))
        out["az_cos"] = np.cos(np.deg2rad(az))
    return out

class FeatureBuilder:
    """Fit scalar features and current-state/trend PCA on training data only."""
    def __init__(self, n_pca_current=8, n_pca_trend=8):
        self.n_pca_current = n_pca_current
        self.n_pca_trend = n_pca_trend
        self.scaler_current = StandardScaler()
        self.scaler_trend = StandardScaler()
        self.pca_current = PCA(n_components=n_pca_current, random_state=RANDOM_SEED)
        self.pca_trend = PCA(n_components=n_pca_trend, random_state=RANDOM_SEED)

    def fit(self, X, elev):
        current = X[:, -1, :]
        trend = X[:, -1, :] - X[:, 0, :]
        self.pca_current.fit(self.scaler_current.fit_transform(current))
        self.pca_trend.fit(self.scaler_trend.fit_transform(trend))
        return self

    def transform(self, X, elev, meta_df=None):
        base = scalar_physical_features(X, elev).reset_index(drop=True)
        if meta_df is not None:
            base = pd.concat([base, temporal_context_features(meta_df)], axis=1)
        current = X[:, -1, :]
        trend = X[:, -1, :] - X[:, 0, :]
        pc_cur = self.pca_current.transform(self.scaler_current.transform(current))
        pc_tr = self.pca_trend.transform(self.scaler_trend.transform(trend))
        for i in range(pc_cur.shape[1]):
            base[f"pc_current_{i+1:02d}"] = pc_cur[:, i]
        for i in range(pc_tr.shape[1]):
            base[f"pc_trend_{i+1:02d}"] = pc_tr[:, i]
        return base

fb = FeatureBuilder(n_pca_current=8, n_pca_trend=8).fit(X_seq[train_idx], X_elev[train_idx])
F_train = fb.transform(X_seq[train_idx], X_elev[train_idx], meta.iloc[train_idx])
F_val = fb.transform(X_seq[val_idx], X_elev[val_idx], meta.iloc[val_idx])
F_test = fb.transform(X_seq[test_idx], X_elev[test_idx], meta.iloc[test_idx])

y_train, y_val, y_test = y[train_idx], y[val_idx], y[test_idx]
print(F_train.shape)
F_train.head()


nan_report = pd.DataFrame({
    "train_nan": F_train.isna().sum(),
    "val_nan": F_val.isna().sum(),
    "test_nan": F_test.isna().sum(),
})
nan_report = nan_report[(nan_report["train_nan"] > 0) | (nan_report["val_nan"] > 0) | (nan_report["test_nan"] > 0)]
if len(nan_report):
    print('Features containing NaN before imputation:')
    display(nan_report.sort_values(["val_nan", "test_nan", "train_nan"], ascending=False))
else:
    print('No NaN values in tabular features.')


## 1. Allineamento metrologico REAL/FEM

Il file di comparazione usa `Effective` come correzione da applicare. Il dataset ML usa invece l'offset misurato. Di conseguenza, per confrontare il FEM con il target ML si usa:

`fem_error = -fem_correction`

L'allineamento ai timestamp viene verificato tramite entrambi gli offset reali, senza usare elevazione e azimuth arrotondati del file FEM.


In [ ]:
COMPARE_CANDIDATES = ["ComparazioneOffsets_REALvsFEM.csv"]
compare_path = find_data_file(COMPARE_CANDIDATES)
print('REAL/FEM comparison:', compare_path)

cmp_raw = pd.read_csv(compare_path, header=[0, 1])
def comma_numeric(series):
    return pd.to_numeric(series.astype(str).str.replace(",", ".", regex=False), errors="coerce")

comparison = pd.DataFrame({
    "Elevation_FEM_file": comma_numeric(cmp_raw.iloc[:, 0]),
    "Azimuth_FEM_file": comma_numeric(cmp_raw.iloc[:, 1]),
    "EL_effective_correction": comma_numeric(cmp_raw.iloc[:, 2]),
    "EL_FEM_correction": comma_numeric(cmp_raw.iloc[:, 3]),
    "XEL_effective_correction": comma_numeric(cmp_raw.iloc[:, 4]),
    "XEL_FEM_correction": comma_numeric(cmp_raw.iloc[:, 5]),
}).dropna().reset_index(drop=True)

# Measured-error targets have the opposite sign to Effective corrections.
comparison["EL_real_error"] = -comparison["EL_effective_correction"]
comparison["XEL_real_error"] = -comparison["XEL_effective_correction"]
comparison["EL_FEM_error"] = -comparison["EL_FEM_correction"]
comparison["XEL_FEM_error"] = -comparison["XEL_FEM_correction"]

target_rows = real_raw.loc[real_raw["XEl Offset"].notna(), [
    "Timestamp", "SRT Elevation Ange[DEG]", "Azimuth", "El Offset", "XEl Offset"
]].copy()
target_rows["key_el"] = target_rows["El Offset"].round(4)
target_rows["key_xel"] = target_rows["XEl Offset"].round(4)
comparison["key_el"] = comparison["EL_real_error"].round(4)
comparison["key_xel"] = comparison["XEL_real_error"].round(4)

assert not target_rows.duplicated(["key_el", "key_xel"]).any(), 'Duplicate offset keys in the real dataset'
assert not comparison.duplicated(["key_el", "key_xel"]).any(), 'Duplicate offset keys in the FEM comparison'

comparison = comparison.merge(
    target_rows[["key_el", "key_xel", "Timestamp", "SRT Elevation Ange[DEG]", "Azimuth"]],
    on=["key_el", "key_xel"], how="left", validate="one_to_one"
).drop(columns=["key_el", "key_xel"])

alignment_df = pd.DataFrame([{
    "comparison_rows": len(comparison),
    "timestamps_matched": comparison["Timestamp"].notna().sum(),
    "max_abs_xel_sign_check": np.max(np.abs(comparison["XEL_real_error"] + comparison["XEL_effective_correction"])),
    "max_abs_el_sign_check": np.max(np.abs(comparison["EL_real_error"] + comparison["EL_effective_correction"])),
    "elevation_rounding_median_deg": np.median(np.abs(comparison["Elevation_FEM_file"] - comparison["SRT Elevation Ange[DEG]"])),
}])
display(alignment_df)
assert comparison["Timestamp"].notna().all(), 'Some FEM comparisons could not be aligned'


paired = meta.reset_index(names="window_index").merge(
    comparison, on="Timestamp", how="inner", validate="one_to_one"
    , suffixes=("_window", "_comparison")
).sort_values("Timestamp").reset_index(drop=True)
pair_idx = paired["window_index"].to_numpy(dtype=int)
X_pair = X_seq[pair_idx]
X_elev_pair = X_elev[pair_idx]
paired["Elevation"] = paired["SRT Elevation Ange[DEG]"]
paired["Azimuth"] = paired["Azimuth_window"].fillna(paired["Azimuth_comparison"])

print('Total comparisons:', len(comparison), '| with clean thermal windows:', len(paired))
print("Range:", paired["Timestamp"].min(), "->", paired["Timestamp"].max())


In [ ]:
audit_rows = []
for axis in ["XEL", "EL"]:
    real = comparison[f"{axis}_real_error"].to_numpy()
    fem = comparison[f"{axis}_FEM_error"].to_numpy()
    residual = real - fem
    audit_rows.append({
        "axis": axis, "count": len(real),
        "real_mean": real.mean(), "real_std": real.std(),
        "fem_mean": fem.mean(), "fem_std": fem.std(),
        "raw_fem_MAE": mean_absolute_error(real, fem),
        "raw_fem_RMSE": np.sqrt(mean_squared_error(real, fem)),
        "raw_fem_R2": r2_score(real, fem),
        "within_5_%": 100 * np.mean(np.abs(residual) <= SPEC_ARCSEC),
        "pearson": np.corrcoef(real, fem)[0, 1],
        "spearman": pd.Series(real).corr(pd.Series(fem), method="spearman"),
        "residual_bias": residual.mean(),
        "residual_p90_abs": np.quantile(np.abs(residual), .90),
    })
real_fem_audit_df = pd.DataFrame(audit_rows)
display(real_fem_audit_df)

fig, axs = plt.subplots(2, 3, figsize=(16, 9))
for row, axis in enumerate(["XEL", "EL"]):
    real = comparison[f"{axis}_real_error"]
    fem = comparison[f"{axis}_FEM_error"]
    residual = real - fem
    lim = np.nanmax(np.abs(np.r_[real, fem]))
    axs[row, 0].scatter(real, fem, c=comparison["SRT Elevation Ange[DEG]"], cmap="viridis", s=22, alpha=.75)
    axs[row, 0].plot([-lim, lim], [-lim, lim], "k--", lw=1)
    axs[row, 0].set(xlabel=f"{axis} observed [arcsec]", ylabel=f"{axis} FEM [arcsec]", title=f"{axis}: REAL vs FEM")
    axs[row, 1].hist(real, bins=45, alpha=.55, density=True, label="REAL")
    axs[row, 1].hist(fem, bins=45, alpha=.55, density=True, label="FEM")
    axs[row, 1].set(title=f"{axis}: distributions", xlabel="Offset [arcsec]"); axs[row, 1].legend()
    sc = axs[row, 2].scatter(comparison["SRT Elevation Ange[DEG]"], residual, c=comparison["Azimuth"], cmap="twilight", s=22, alpha=.75)
    axs[row, 2].axhline(0, color="k", ls="--", lw=1)
    axs[row, 2].set(xlabel='Elevation [deg]', ylabel="REAL - FEM [arcsec]", title=f"{axis}: geometric residual")
for ax in axs.ravel(): ax.grid(True, alpha=.25)
plt.tight_layout(); plt.show()


## 2. Calibrazioni semplici e test cronologico

Questa sezione verifica se il gap è riducibile con bias, scala, geometria o un polinomio regolarizzato. Questi modelli sono diagnostici: se non superano una baseline mediana, il FEM non contiene abbastanza informazione sul target reale.


In [ ]:
p_train, p_val, p_test = chronological_split(paired)
split_map = {"train": p_train, "val": p_val, "test": p_test}
print({k: len(v) for k, v in split_map.items()})
for name, idx in split_map.items():
    print(name, paired.iloc[idx[0]]["Timestamp"], "->", paired.iloc[idx[-1]]["Timestamp"])

from sklearn.preprocessing import PolynomialFeatures, RobustScaler

def regression_metrics(axis, model, split, y_true, y_pred, family):
    err = np.asarray(y_true) - np.asarray(y_pred)
    return {
        "axis": axis, "family": family, "model": model, "split": split,
        "count": len(err), "MAE": np.mean(np.abs(err)),
        "RMSE": np.sqrt(np.mean(err**2)),
        "R2": r2_score(y_true, y_pred),
        "within_5_%": 100*np.mean(np.abs(err) <= SPEC_ARCSEC),
        "p90_abs_error": np.quantile(np.abs(err), .90),
    }

paired["sin_el"] = np.sin(np.deg2rad(paired["Elevation"]))
paired["cos_el"] = np.cos(np.deg2rad(paired["Elevation"]))
paired["az_sin"] = np.sin(np.deg2rad(paired["Azimuth"]))
paired["az_cos"] = np.cos(np.deg2rad(paired["Azimuth"]))
geometry_cols = ["Elevation", "sin_el", "cos_el", "az_sin", "az_cos"]


In [ ]:
simple_rows = []
simple_predictions = {}
for axis in ["XEL", "EL"]:
    y_axis = paired[f"{axis}_real_error"].to_numpy()
    fem_col = f"{axis}_FEM_error"
    feature_sets = {
        "affine_FEM": [fem_col],
        "FEM_geometry": [fem_col] + geometry_cols,
    }
    models = {
        "affine_FEM": Pipeline([("imputer", SimpleImputer(strategy="median")), ("scale", StandardScaler()), ("reg", RidgeCV(alphas=np.logspace(-5, 5, 31)))]),
        "FEM_geometry": Pipeline([("imputer", SimpleImputer(strategy="median")), ("scale", StandardScaler()), ("reg", RidgeCV(alphas=np.logspace(-5, 5, 31)))]),
        "FEM_geometry_poly2": Pipeline([("imputer", SimpleImputer(strategy="median")), ("poly", PolynomialFeatures(2, include_bias=False)), ("scale", StandardScaler()), ("reg", RidgeCV(alphas=np.logspace(-4, 6, 31)))]),
    }
    for split in ["val", "test"]:
        idx = split_map[split]
        baselines = {
            "raw_FEM": paired[fem_col].to_numpy()[idx],
            "zero": np.zeros(len(idx)),
            "train_median": np.full(len(idx), np.median(y_axis[p_train])),
            "FEM_plus_train_median_residual": paired[fem_col].to_numpy()[idx] + np.median(y_axis[p_train] - paired[fem_col].to_numpy()[p_train]),
        }
        for name, pred in baselines.items():
            simple_rows.append(regression_metrics(axis, name, split, y_axis[idx], pred, "simple_calibration"))
            simple_predictions[(axis, name, split)] = pred
    for name, model in models.items():
        cols = [fem_col] + geometry_cols
        if name == "affine_FEM": cols = [fem_col]
        m = clone(model).fit(paired.iloc[p_train][cols], y_axis[p_train])
        for split in ["val", "test"]:
            idx = split_map[split]; pred = m.predict(paired.iloc[idx][cols])
            simple_rows.append(regression_metrics(axis, name, split, y_axis[idx], pred, "simple_calibration"))
            simple_predictions[(axis, name, split)] = pred

simple_calibration_df = pd.DataFrame(simple_rows).sort_values(["axis", "split", "MAE"])
display(simple_calibration_df)


## 3. Ablation multi-fidelity sui 671 campioni puliti

Confronti sullo stesso spazio temporale:
- solo stato termico reale;
- solo FEM e geometria;
- termico + FEM;
- correzione residuale `FEM + Delta(temperature, geometria, FEM)`.

La PCA e ogni scaler sono fittati esclusivamente sul training.


In [ ]:
fb_pair = FeatureBuilder(n_pca_current=8, n_pca_trend=8).fit(X_pair[p_train], X_elev_pair[p_train])
F_pair = fb_pair.transform(X_pair, X_elev_pair, paired)
F_pair = F_pair.reset_index(drop=True)
F_pair["XEL_FEM_error"] = paired["XEL_FEM_error"].to_numpy()
F_pair["EL_FEM_error"] = paired["EL_FEM_error"].to_numpy()

thermal_cols = [c for c in F_pair.columns if c not in ["XEL_FEM_error", "EL_FEM_error"]]
print('Thermal/geometric features:', len(thermal_cols), '| with FEM:', F_pair.shape[1])
display(F_pair.head())

candidate_models = {
    "ridge": Pipeline([("imputer", SimpleImputer(strategy="median")), ("scale", StandardScaler()), ("reg", RidgeCV(alphas=np.logspace(-4, 5, 30)))]),
    "huber": Pipeline([("imputer", SimpleImputer(strategy="median")), ("scale", RobustScaler()), ("reg", HuberRegressor(epsilon=1.5, alpha=1.0, max_iter=4000))]),
    "random_forest": Pipeline([("imputer", SimpleImputer(strategy="median")), ("reg", RandomForestRegressor(n_estimators=600, min_samples_leaf=8, max_features="sqrt", random_state=RANDOM_SEED, n_jobs=-1))]),
    "hist_gradient_boosting": Pipeline([("imputer", SimpleImputer(strategy="median")), ("reg", HistGradientBoostingRegressor(loss="absolute_error", max_iter=250, learning_rate=.03, max_leaf_nodes=12, min_samples_leaf=15, l2_regularization=3, random_state=RANDOM_SEED))]),
}


In [ ]:
ablation_rows = []
ablation_predictions = {}
for axis in ["XEL", "EL"]:
    y_axis = paired[f"{axis}_real_error"].to_numpy()
    fem_col = f"{axis}_FEM_error"
    fem = paired[fem_col].to_numpy()
    feature_sets = {
        "thermal_only": thermal_cols,
        "FEM_geometry": [fem_col] + [c for c in geometry_cols if c in F_pair.columns],
        "thermal_plus_FEM": thermal_cols + [fem_col],
    }
    for family, cols in feature_sets.items():
        for model_name, model in candidate_models.items():
            m = clone(model).fit(F_pair.iloc[p_train][cols], y_axis[p_train])
            for split in ["val", "test"]:
                idx=split_map[split]; pred=m.predict(F_pair.iloc[idx][cols])
                ablation_rows.append(regression_metrics(axis, model_name, split, y_axis[idx], pred, family))
                ablation_predictions[(axis, family, model_name, split)] = pred

    # Residual learning: REAL = FEM + predicted discrepancy.
    residual_target = y_axis - fem
    residual_cols = thermal_cols + [fem_col]
    for model_name, model in candidate_models.items():
        m = clone(model).fit(F_pair.iloc[p_train][residual_cols], residual_target[p_train])
        for split in ["val", "test"]:
            idx=split_map[split]
            delta_pred=m.predict(F_pair.iloc[idx][residual_cols]); pred=fem[idx]+delta_pred
            ablation_rows.append(regression_metrics(axis, model_name, split, y_axis[idx], pred, "FEM_plus_residual"))
            ablation_predictions[(axis, "FEM_plus_residual", model_name, split)] = pred

ablation_df = pd.DataFrame(ablation_rows).sort_values(["axis", "split", "MAE"])
display(ablation_df)

# Select on validation before evaluating the corresponding historical test.
selected_rows=[]
for axis in ["XEL", "EL"]:
    val_axis=ablation_df[(ablation_df["axis"]==axis)&(ablation_df["split"]=="val")]
    best=val_axis.loc[val_axis.MAE.idxmin()]
    test=ablation_df[(ablation_df["axis"]==axis)&(ablation_df["split"]=="test")&(ablation_df["family"]==best.family)&(ablation_df["model"]==best.model)].iloc[0]
    selected_rows.append({"axis":axis,"selected_family":best.family,"selected_model":best.model,"validation_MAE":best.MAE,"test_MAE":test.MAE,"test_within_5_%":test["within_5_%"],"test_p90_abs_error":test.p90_abs_error})
selected_ablation_df=pd.DataFrame(selected_rows)
display(selected_ablation_df)


## 4. Stabilità temporale e localizzazione del gap

Il singolo holdout finale può dipendere fortemente dalla stagione. La blocked cross-validation seguente confronta le famiglie senza mescolare futuro e passato.


In [ ]:
cv_rows=[]
n=len(paired)
fold_bounds=[]
for frac in [.50,.625,.75,.875]:
    end_train=int(n*frac); end_eval=min(n,end_train+max(25,int(n*.125)))
    if end_eval>end_train: fold_bounds.append((end_train,end_eval))

cv_model=Pipeline([("imputer",SimpleImputer(strategy="median")),("scale",StandardScaler()),("reg",RidgeCV(alphas=np.logspace(-4,5,30)))])
for axis in ["XEL","EL"]:
    y_axis=paired[f"{axis}_real_error"].to_numpy(); fem=paired[f"{axis}_FEM_error"].to_numpy(); fem_col=f"{axis}_FEM_error"
    families={"thermal_only":thermal_cols,"thermal_plus_FEM":thermal_cols+[fem_col]}
    for fold,(end_train,end_eval) in enumerate(fold_bounds,1):
        tr=np.arange(end_train); ev=np.arange(end_train,end_eval)
        for family,cols in families.items():
            m=clone(cv_model).fit(F_pair.iloc[tr][cols],y_axis[tr]); pred=m.predict(F_pair.iloc[ev][cols])
            cv_rows.append({"fold":fold,"eval_start":paired.iloc[ev[0]].Timestamp,"eval_end":paired.iloc[ev[-1]].Timestamp,**regression_metrics(axis,"ridge",f"fold_{fold}",y_axis[ev],pred,family)})
        residual=y_axis-fem; cols=thermal_cols+[fem_col]
        m=clone(cv_model).fit(F_pair.iloc[tr][cols],residual[tr]); pred=fem[ev]+m.predict(F_pair.iloc[ev][cols])
        cv_rows.append({"fold":fold,"eval_start":paired.iloc[ev[0]].Timestamp,"eval_end":paired.iloc[ev[-1]].Timestamp,**regression_metrics(axis,"ridge",f"fold_{fold}",y_axis[ev],pred,"FEM_plus_residual")})

multifidelity_cv_df=pd.DataFrame(cv_rows)
display(multifidelity_cv_df)
display(multifidelity_cv_df.groupby(["axis","family"])[["MAE","within_5_%","p90_abs_error"]].agg(["mean","std","min","max"]))


In [ ]:
diagnostic_rows=[]
for axis in ["XEL","EL"]:
    d=paired[["Timestamp","Elevation","Azimuth",f"{axis}_real_error",f"{axis}_FEM_error"]].copy()
    d["axis"]=axis; d["residual"]=d[f"{axis}_real_error"]-d[f"{axis}_FEM_error"]
    d["abs_residual"]=d.residual.abs(); d["abs_target"]=d[f"{axis}_real_error"].abs()
    d["elev_band"]=pd.cut(d.Elevation,[0,20,30,45,60,75,90],include_lowest=True)
    d["az_band"]=pd.cut(d.Azimuth,[0,45,90,135,180,225,270,315,360],include_lowest=True)
    d["month"]=d.Timestamp.dt.month
    diagnostic_rows.append(d)
residual_diagnostic_df=pd.concat(diagnostic_rows,ignore_index=True)

print('Residual by elevation')
display(residual_diagnostic_df.groupby(["axis","elev_band"],observed=False).agg(count=("residual","size"),bias=("residual","mean"),MAE=("abs_residual","mean"),p90=("abs_residual",lambda x:x.quantile(.9))))
print('Residual by azimuth')
display(residual_diagnostic_df.groupby(["axis","az_band"],observed=False).agg(count=("residual","size"),bias=("residual","mean"),MAE=("abs_residual","mean")))
print('Residual by month')
display(residual_diagnostic_df.groupby(["axis","month"]).agg(count=("residual","size"),bias=("residual","mean"),MAE=("abs_residual","mean"),p90=("abs_residual",lambda x:x.quantile(.9))))


## 5. Correlazione REAL/FEM al netto della geometria

Questo test rimuove separatamente da REAL e FEM il comportamento medio regolare in funzione di elevazione e azimuth. L'elevazione è modellata con spline regolarizzate; l'azimuth, che è circolare, con tre armoniche seno/coseno.

Tutte le trasformazioni sono fittate esclusivamente sul training. Il test è diagnostico: verifica se, a parità di geometria, gli scostamenti FEM dal proprio comportamento medio seguono gli scostamenti reali.


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import SplineTransformer

AZ_HARMONICS = 3
EL_SPLINE_KNOTS = 6

def geometry_feature_frame(df):
    out = pd.DataFrame(index=df.index)
    out["Elevation"] = pd.to_numeric(df["Elevation"], errors="coerce")
    az_rad = np.deg2rad(pd.to_numeric(df["Azimuth"], errors="coerce"))
    for harmonic in range(1, AZ_HARMONICS + 1):
        out[f"az_sin_{harmonic}"] = np.sin(harmonic * az_rad)
        out[f"az_cos_{harmonic}"] = np.cos(harmonic * az_rad)
    return out

az_fourier_cols = [f"az_{kind}_{h}" for h in range(1, AZ_HARMONICS + 1) for kind in ["sin", "cos"]]

def make_smooth_geometry_model():
    elevation_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("spline", SplineTransformer(n_knots=EL_SPLINE_KNOTS, degree=3, include_bias=False)),
        ("scale", StandardScaler()),
    ])
    azimuth_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scale", StandardScaler()),
    ])
    features = ColumnTransformer([
        ("elevation_spline", elevation_pipe, ["Elevation"]),
        ("azimuth_fourier", azimuth_pipe, az_fourier_cols),
    ])
    return Pipeline([
        ("geometry", features),
        ("reg", RidgeCV(alphas=np.logspace(-4, 5, 40))),
    ])

G_pair = geometry_feature_frame(paired)
geometry_residuals = {}
geometry_correlation_rows = []

for axis in ["XEL", "EL"]:
    y_real = paired[f"{axis}_real_error"].to_numpy(dtype=float)
    y_fem = paired[f"{axis}_FEM_error"].to_numpy(dtype=float)

    g_real = make_smooth_geometry_model().fit(G_pair.iloc[p_train], y_real[p_train])
    g_fem = make_smooth_geometry_model().fit(G_pair.iloc[p_train], y_fem[p_train])
    pred_geometry_real = g_real.predict(G_pair)
    pred_geometry_fem = g_fem.predict(G_pair)
    r_real = y_real - pred_geometry_real
    r_fem = y_fem - pred_geometry_fem

    geometry_residuals[axis] = {
        "y_real": y_real, "y_fem": y_fem,
        "g_real": pred_geometry_real, "g_fem": pred_geometry_fem,
        "r_real": r_real, "r_fem": r_fem,
    }

    for split, idx in split_map.items():
        pearson = np.corrcoef(r_real[idx], r_fem[idx])[0, 1]
        spearman = pd.Series(r_real[idx]).corr(pd.Series(r_fem[idx]), method="spearman")
        geometry_correlation_rows.append({
            "axis": axis, "split": split, "count": len(idx),
            "pearson_r_real_r_FEM": pearson,
            "spearman_r_real_r_FEM": spearman,
            "real_residual_std": np.std(r_real[idx]),
            "FEM_residual_std": np.std(r_fem[idx]),
            "geometry_only_MAE": mean_absolute_error(y_real[idx], pred_geometry_real[idx]),
        })

geometry_residual_correlation_df = pd.DataFrame(geometry_correlation_rows)
display(geometry_residual_correlation_df)

fig, axs = plt.subplots(1, 2, figsize=(13, 5))
for ax, axis in zip(axs, ["XEL", "EL"]):
    d = geometry_residuals[axis]
    ax.scatter(d["r_fem"][p_test], d["r_real"][p_test], c=paired.iloc[p_test]["Elevation"], cmap="viridis", alpha=.8)
    ax.axhline(0, color="k", ls="--", lw=1); ax.axvline(0, color="k", ls="--", lw=1)
    ax.set(xlabel='FEM residual after geometry removal [arcsec]', ylabel='REAL residual after geometry removal [arcsec]', title=f"{axis}: partial correlation on the test")
    ax.grid(True, alpha=.25)
plt.tight_layout(); plt.show()


In [ ]:
residual_calibration_rows = []
residual_calibration_predictions = {}

residual_calibrators = {
    "ridge": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scale", StandardScaler()),
        ("reg", RidgeCV(alphas=np.logspace(-5, 5, 40))),
    ]),
    "huber": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scale", RobustScaler()),
        ("reg", HuberRegressor(epsilon=1.5, alpha=1.0, max_iter=4000)),
    ]),
}

for axis in ["XEL", "EL"]:
    d = geometry_residuals[axis]
    X_residual = d["r_fem"].reshape(-1, 1)

    for model_name, model in residual_calibrators.items():
        calibrator = clone(model).fit(X_residual[p_train], d["r_real"][p_train])
        for split in ["val", "test"]:
            idx = split_map[split]
            r_real_hat = calibrator.predict(X_residual[idx])
            final_prediction = d["g_real"][idx] + r_real_hat
            geometry_prediction = d["g_real"][idx]
            mae_geometry = mean_absolute_error(d["y_real"][idx], geometry_prediction)
            mae_with_fem = mean_absolute_error(d["y_real"][idx], final_prediction)
            residual_calibration_rows.append({
                "axis": axis, "model": model_name, "split": split, "count": len(idx),
                "residual_calibration_MAE": mean_absolute_error(d["r_real"][idx], r_real_hat),
                "geometry_only_MAE": mae_geometry,
                "geometry_plus_r_FEM_MAE": mae_with_fem,
                "delta_MAE_with_minus_without_FEM": mae_with_fem - mae_geometry,
                "improvement_%": 100 * (mae_geometry - mae_with_fem) / mae_geometry,
                "within_5_geometry_%": 100 * np.mean(np.abs(d["y_real"][idx] - geometry_prediction) <= SPEC_ARCSEC),
                "within_5_plus_r_FEM_%": 100 * np.mean(np.abs(d["y_real"][idx] - final_prediction) <= SPEC_ARCSEC),
            })
            residual_calibration_predictions[(axis, model_name, split)] = final_prediction

geometry_residual_calibration_df = pd.DataFrame(residual_calibration_rows)
display(geometry_residual_calibration_df.sort_values(["axis", "split", "geometry_plus_r_FEM_MAE"]))


selected_geometry_residual_rows = []
for axis in ["XEL", "EL"]:
    val_rows = geometry_residual_calibration_df[
        (geometry_residual_calibration_df["axis"] == axis) &
        (geometry_residual_calibration_df["split"] == "val")
    ]
    best_val = val_rows.loc[val_rows["geometry_plus_r_FEM_MAE"].idxmin()]
    test_row = geometry_residual_calibration_df[
        (geometry_residual_calibration_df["axis"] == axis) &
        (geometry_residual_calibration_df["split"] == "test") &
        (geometry_residual_calibration_df["model"] == best_val["model"])
    ].iloc[0]
    selected_geometry_residual_rows.append({
        "axis": axis, "selected_model": best_val["model"],
        "validation_MAE_with_r_FEM": best_val["geometry_plus_r_FEM_MAE"],
        "test_geometry_only_MAE": test_row["geometry_only_MAE"],
        "test_geometry_plus_r_FEM_MAE": test_row["geometry_plus_r_FEM_MAE"],
        "test_delta_MAE": test_row["delta_MAE_with_minus_without_FEM"],
        "test_improvement_%": test_row["improvement_%"],
    })
selected_geometry_residual_df = pd.DataFrame(selected_geometry_residual_rows)
display(selected_geometry_residual_df)


In [ ]:
geometry_residual_block_rows = []
n_samples = len(paired)
geometry_fold_bounds = []
for train_fraction in [.50, .625, .75, .875]:
    train_end = int(n_samples * train_fraction)
    eval_end = min(n_samples, train_end + max(25, int(n_samples * .125)))
    if eval_end > train_end:
        geometry_fold_bounds.append((train_end, eval_end))

for axis in ["XEL", "EL"]:
    y_real_all = paired[f"{axis}_real_error"].to_numpy(dtype=float)
    y_fem_all = paired[f"{axis}_FEM_error"].to_numpy(dtype=float)

    for fold, (train_end, eval_end) in enumerate(geometry_fold_bounds, start=1):
        tr = np.arange(train_end)
        ev = np.arange(train_end, eval_end)
        g_real = make_smooth_geometry_model().fit(G_pair.iloc[tr], y_real_all[tr])
        g_fem = make_smooth_geometry_model().fit(G_pair.iloc[tr], y_fem_all[tr])
        real_geometry_tr = g_real.predict(G_pair.iloc[tr])
        fem_geometry_tr = g_fem.predict(G_pair.iloc[tr])
        real_geometry_ev = g_real.predict(G_pair.iloc[ev])
        fem_geometry_ev = g_fem.predict(G_pair.iloc[ev])
        r_real_tr = y_real_all[tr] - real_geometry_tr
        r_fem_tr = y_fem_all[tr] - fem_geometry_tr
        r_real_ev = y_real_all[ev] - real_geometry_ev
        r_fem_ev = y_fem_all[ev] - fem_geometry_ev

        calibrator = clone(residual_calibrators["ridge"]).fit(r_fem_tr.reshape(-1, 1), r_real_tr)
        r_hat_ev = calibrator.predict(r_fem_ev.reshape(-1, 1))
        pred_geometry = real_geometry_ev
        pred_plus_fem = real_geometry_ev + r_hat_ev
        mae_geometry = mean_absolute_error(y_real_all[ev], pred_geometry)
        mae_plus_fem = mean_absolute_error(y_real_all[ev], pred_plus_fem)

        geometry_residual_block_rows.append({
            "axis": axis, "fold": fold, "n_train": len(tr), "n_eval": len(ev),
            "eval_start": paired.iloc[ev[0]]["Timestamp"],
            "eval_end": paired.iloc[ev[-1]]["Timestamp"],
            "pearson_r_real_r_FEM": np.corrcoef(r_real_ev, r_fem_ev)[0, 1],
            "spearman_r_real_r_FEM": pd.Series(r_real_ev).corr(pd.Series(r_fem_ev), method="spearman"),
            "geometry_only_MAE": mae_geometry,
            "geometry_plus_r_FEM_MAE": mae_plus_fem,
            "delta_MAE_with_minus_without_FEM": mae_plus_fem - mae_geometry,
            "improvement_%": 100 * (mae_geometry - mae_plus_fem) / mae_geometry,
        })

geometry_residual_blocked_df = pd.DataFrame(geometry_residual_block_rows)
display(geometry_residual_blocked_df)
print('Temporal-stability summary')
display(geometry_residual_blocked_df.groupby("axis").agg(
    folds=("fold", "count"),
    pearson_mean=("pearson_r_real_r_FEM", "mean"),
    pearson_std=("pearson_r_real_r_FEM", "std"),
    delta_MAE_mean=("delta_MAE_with_minus_without_FEM", "mean"),
    delta_MAE_std=("delta_MAE_with_minus_without_FEM", "std"),
    folds_improved=("delta_MAE_with_minus_without_FEM", lambda x: int((x < 0).sum())),
))

geometry_residual_correlation_df.to_csv("fem_geometry_residual_correlations.csv", index=False)
geometry_residual_calibration_df.to_csv("fem_geometry_residual_calibration.csv", index=False)
selected_geometry_residual_df.to_csv("fem_geometry_residual_selected.csv", index=False)
geometry_residual_blocked_df.to_csv("fem_geometry_residual_blocked_cv.csv", index=False)


## 6. Trend robusti e audit dell'interpolazione termica

La prima parte confronta trend REAL/FEM dopo la rimozione della geometria, usando mediane locali entro sessioni e senza attraversare i confini train/validation/test. La valutazione è a livello di trend, non punto per punto.

La seconda parte esegue una leave-one-sensor-out data-driven. Per ogni sensore sceglie i `k` sensori più correlati sul training e verifica su validation/test la ricostruzione della temperatura e dei gradienti spaziali. È un proxy della ricostruibilità del campo termico: senza coordinate e connettività FEM non replica il metodo topologico adottato nel simulatore.


In [ ]:
TREND_SESSION_GAP_HOURS = 6.0
TREND_WINDOWS = [1, 3, 5]

def subset_session_median(values, indices, timestamps, window):
    """Compute offline centered medians without crossing session gaps."""
    indices = np.asarray(indices, dtype=int)
    if len(indices) == 0:
        return np.array([], dtype=float)
    sub = pd.DataFrame({
        "position": np.arange(len(indices)),
        "Timestamp": pd.to_datetime(timestamps.iloc[indices]).to_numpy(),
        "value": np.asarray(values, dtype=float)[indices],
    }).sort_values("Timestamp").reset_index(drop=True)
    gaps = sub["Timestamp"].diff().dt.total_seconds().div(3600).fillna(np.inf)
    sub["session"] = (gaps > TREND_SESSION_GAP_HOURS).cumsum()
    sub["filtered"] = sub.groupby("session", sort=False)["value"].transform(
        lambda s: s.rolling(window=window, center=True, min_periods=1).median()
    )
    out = np.empty(len(indices), dtype=float)
    out[sub["position"].to_numpy(dtype=int)] = sub["filtered"].to_numpy(dtype=float)
    return out

trend_split_rows=[]
trend_filtered={}
for axis in ["XEL","EL"]:
    r_real=geometry_residuals[axis]["r_real"]
    r_fem=geometry_residuals[axis]["r_fem"]
    for window in TREND_WINDOWS:
        full_real=np.full(len(paired),np.nan); full_fem=np.full(len(paired),np.nan)
        for split,idx in split_map.items():
            rr=subset_session_median(r_real,idx,paired["Timestamp"],window)
            rf=subset_session_median(r_fem,idx,paired["Timestamp"],window)
            full_real[idx]=rr; full_fem[idx]=rf
            trend_split_rows.append({
                "axis":axis,"window":window,"split":split,"count":len(idx),
                "pearson_trend":np.corrcoef(rr,rf)[0,1],
                "spearman_trend":pd.Series(rr).corr(pd.Series(rf),method="spearman"),
                "trend_MAE_uncalibrated":mean_absolute_error(rr,rf),
                "real_trend_std":np.std(rr),"FEM_trend_std":np.std(rf),
                "real_variance_retained_%":100*np.var(rr)/np.var(r_real[idx]) if np.var(r_real[idx])>0 else np.nan,
                "FEM_variance_retained_%":100*np.var(rf)/np.var(r_fem[idx]) if np.var(r_fem[idx])>0 else np.nan,
            })
        trend_filtered[(axis,window,"real")]=full_real
        trend_filtered[(axis,window,"fem")]=full_fem

trend_filter_comparison_df=pd.DataFrame(trend_split_rows)
display(trend_filter_comparison_df)

fig,axs=plt.subplots(2,3,figsize=(16,9),sharex=False)
for row,axis in enumerate(["XEL","EL"]):
    for col,window in enumerate(TREND_WINDOWS):
        rr=trend_filtered[(axis,window,"real")][p_test]
        rf=trend_filtered[(axis,window,"fem")][p_test]
        axs[row,col].scatter(rf,rr,c=paired.iloc[p_test]["Elevation"],cmap="viridis",alpha=.75,s=24)
        axs[row,col].axhline(0,color="k",ls="--",lw=1); axs[row,col].axvline(0,color="k",ls="--",lw=1)
        axs[row,col].set(title=f"{axis}, median {window}",xlabel='FEM residual trend',ylabel='REAL residual trend')
        axs[row,col].grid(True,alpha=.25)
plt.tight_layout(); plt.show()


In [ ]:
trend_calibration_rows=[]
trend_selected_rows=[]

for axis in ["XEL","EL"]:
    for window in TREND_WINDOWS:
        real_trend=trend_filtered[(axis,window,"real")]
        fem_trend=trend_filtered[(axis,window,"fem")]
        models={
            "ridge":Pipeline([("scale",StandardScaler()),("reg",RidgeCV(alphas=np.logspace(-5,5,40)))]),
            "huber":Pipeline([("scale",RobustScaler()),("reg",HuberRegressor(epsilon=1.5,alpha=1.0,max_iter=4000))]),
        }
        for name,model in models.items():
            m=clone(model).fit(fem_trend[p_train].reshape(-1,1),real_trend[p_train])
            for split in ["val","test"]:
                idx=split_map[split]
                pred=m.predict(fem_trend[idx].reshape(-1,1))
                baseline=np.full(len(idx),np.median(real_trend[p_train]))
                mae_base=mean_absolute_error(real_trend[idx],baseline)
                mae_pred=mean_absolute_error(real_trend[idx],pred)
                trend_calibration_rows.append({
                    "axis":axis,"window":window,"model":name,"split":split,"count":len(idx),
                    "baseline_trend_MAE":mae_base,"calibrated_FEM_trend_MAE":mae_pred,
                    "delta_MAE":mae_pred-mae_base,
                    "improvement_%":100*(mae_base-mae_pred)/mae_base if mae_base>0 else np.nan,
                    "pearson_trend":np.corrcoef(real_trend[idx],fem_trend[idx])[0,1],
                })

trend_calibration_df=pd.DataFrame(trend_calibration_rows)
display(trend_calibration_df.sort_values(["axis","split","calibrated_FEM_trend_MAE"]))


for axis in ["XEL","EL"]:
    val=trend_calibration_df[(trend_calibration_df["axis"]==axis)&(trend_calibration_df["split"]=="val")]
    best=val.loc[val.calibrated_FEM_trend_MAE.idxmin()]
    test=trend_calibration_df[(trend_calibration_df["axis"]==axis)&(trend_calibration_df["split"]=="test")&(trend_calibration_df["window"]==best.window)&(trend_calibration_df["model"]==best.model)].iloc[0]
    trend_selected_rows.append({
        "axis":axis,"selected_window":int(best.window),"selected_model":best.model,
        "validation_MAE":best.calibrated_FEM_trend_MAE,
        "test_baseline_MAE":test.baseline_trend_MAE,"test_FEM_trend_MAE":test.calibrated_FEM_trend_MAE,
        "test_delta_MAE":test.delta_MAE,"test_improvement_%":test["improvement_%"],
    })
trend_selected_df=pd.DataFrame(trend_selected_rows)
display(trend_selected_df)


In [ ]:
TREND_OBJECTIVE_LAMBDA=0.25
trend_block_rows=[]

for axis in ["XEL","EL"]:
    y_real_all=paired[f"{axis}_real_error"].to_numpy(float)
    y_fem_all=paired[f"{axis}_FEM_error"].to_numpy(float)
    selected_window=int(trend_selected_df.loc[trend_selected_df["axis"]==axis,"selected_window"].iloc[0])

    for fold,(train_end,eval_end) in enumerate(geometry_fold_bounds,1):
        tr=np.arange(train_end); ev=np.arange(train_end,eval_end)
        g_real=make_smooth_geometry_model().fit(G_pair.iloc[tr],y_real_all[tr])
        g_fem=make_smooth_geometry_model().fit(G_pair.iloc[tr],y_fem_all[tr])
        r_real_tr=y_real_all[tr]-g_real.predict(G_pair.iloc[tr]); r_fem_tr=y_fem_all[tr]-g_fem.predict(G_pair.iloc[tr])
        r_real_ev=y_real_all[ev]-g_real.predict(G_pair.iloc[ev]); r_fem_ev=y_fem_all[ev]-g_fem.predict(G_pair.iloc[ev])
        real_tr=subset_session_median(r_real_tr,np.arange(len(tr)),paired.iloc[tr]["Timestamp"].reset_index(drop=True),selected_window)
        fem_tr=subset_session_median(r_fem_tr,np.arange(len(tr)),paired.iloc[tr]["Timestamp"].reset_index(drop=True),selected_window)
        real_ev=subset_session_median(r_real_ev,np.arange(len(ev)),paired.iloc[ev]["Timestamp"].reset_index(drop=True),selected_window)
        fem_ev=subset_session_median(r_fem_ev,np.arange(len(ev)),paired.iloc[ev]["Timestamp"].reset_index(drop=True),selected_window)
        calibrator=Pipeline([("scale",StandardScaler()),("reg",RidgeCV(alphas=np.logspace(-5,5,40)))]).fit(fem_tr.reshape(-1,1),real_tr)
        pred=calibrator.predict(fem_ev.reshape(-1,1)); baseline=np.full(len(ev),np.median(real_tr))
        correlation=np.corrcoef(real_ev,fem_ev)[0,1]
        nmae=mean_absolute_error(real_ev,fem_ev)/(np.std(real_ev)+1e-8)
        trend_block_rows.append({
            "axis":axis,"fold":fold,"window":selected_window,"n_train":len(tr),"n_eval":len(ev),
            "eval_start":paired.iloc[ev[0]].Timestamp,"eval_end":paired.iloc[ev[-1]].Timestamp,
            "pearson_trend":correlation,"spearman_trend":pd.Series(real_ev).corr(pd.Series(fem_ev),method="spearman"),
            "baseline_trend_MAE":mean_absolute_error(real_ev,baseline),
            "calibrated_FEM_trend_MAE":mean_absolute_error(real_ev,pred),
            "delta_MAE":mean_absolute_error(real_ev,pred)-mean_absolute_error(real_ev,baseline),
            "amplitude_ratio_FEM_over_REAL":np.std(fem_ev)/(np.std(real_ev)+1e-8),
            "trend_objective_raw":1-correlation+TREND_OBJECTIVE_LAMBDA*nmae,
        })

trend_blocked_df=pd.DataFrame(trend_block_rows)
display(trend_blocked_df)
trend_stability_summary_df=trend_blocked_df.groupby("axis").agg(
    folds=("fold","count"),pearson_mean=("pearson_trend","mean"),pearson_std=("pearson_trend","std"),
    delta_MAE_mean=("delta_MAE","mean"),delta_MAE_std=("delta_MAE","std"),
    folds_improved=("delta_MAE",lambda x:int((x<0).sum())),
    amplitude_ratio_mean=("amplitude_ratio_FEM_over_REAL","mean"),
    objective_mean=("trend_objective_raw","mean"),
).reset_index()
display(trend_stability_summary_df)

trend_filter_comparison_df.to_csv("fem_trend_filter_comparison.csv",index=False)
trend_calibration_df.to_csv("fem_trend_calibration.csv",index=False)
trend_selected_df.to_csv("fem_trend_selected.csv",index=False)
trend_blocked_df.to_csv("fem_trend_blocked_cv.csv",index=False)


In [ ]:
SENSOR_NEIGHBOR_COUNTS=[3,5,10]


sensor_state=X_pair[:,-1,:].astype(float)
train_sensor=sensor_state[p_train]
sensor_mean=np.nanmean(train_sensor,axis=0)
sensor_std=np.nanstd(train_sensor,axis=0)
sensor_std=np.where(sensor_std<1e-6,1.0,sensor_std)
z_train=(train_sensor-sensor_mean)/sensor_std
sensor_corr=np.corrcoef(z_train,rowvar=False)
sensor_corr=np.nan_to_num(sensor_corr,nan=0.0,posinf=0.0,neginf=0.0)
np.fill_diagonal(sensor_corr,-np.inf)

def reconstruct_all_sensors(values,k):
    z=(values-sensor_mean)/sensor_std
    pred=np.empty_like(values,dtype=float)
    neighbors=[]
    for target_sensor in range(values.shape[1]):
        idx=np.argsort(sensor_corr[target_sensor])[::-1][:k]
        weights=np.maximum(sensor_corr[target_sensor,idx],0.0)
        if weights.sum()<=1e-12: weights=np.ones(k)
        weights=weights/weights.sum()
        z_hat=z[:,idx]@weights
        pred[:,target_sensor]=sensor_mean[target_sensor]+sensor_std[target_sensor]*z_hat
        neighbors.append(idx)
    return pred,neighbors

sensor_reconstruction_rows=[]
sensor_predictions={}
sensor_neighbors={}
for k in SENSOR_NEIGHBOR_COUNTS:
    pred,neighbors=reconstruct_all_sensors(sensor_state,k)
    sensor_predictions[k]=pred; sensor_neighbors[k]=neighbors
    for split in ["val","test"]:
        idx=split_map[split]; true=sensor_state[idx]; estimated=pred[idx]
        sensor_mae=np.mean(np.abs(true-estimated),axis=0)
        true_centered=true-true.mean(axis=1,keepdims=True)
        pred_centered=estimated-estimated.mean(axis=1,keepdims=True)
        true_spatial_std=true.std(axis=1); pred_spatial_std=estimated.std(axis=1)
        sensor_reconstruction_rows.append({
            "k_neighbors":k,"split":split,"samples":len(idx),
            "cell_MAE_C":np.mean(np.abs(true-estimated)),
            "cell_RMSE_C":np.sqrt(np.mean((true-estimated)**2)),
            "median_sensor_MAE_C":np.median(sensor_mae),"p90_sensor_MAE_C":np.quantile(sensor_mae,.9),
            "max_sensor_MAE_C":sensor_mae.max(),
            "centered_pattern_RMSE_C":np.sqrt(np.mean((true_centered-pred_centered)**2)),
            "spatial_std_MAE_C":mean_absolute_error(true_spatial_std,pred_spatial_std),
            "spatial_std_correlation":np.corrcoef(true_spatial_std,pred_spatial_std)[0,1],
            "spatial_std_ratio_mean":np.mean(pred_spatial_std/(true_spatial_std+1e-8)),
        })

sensor_interpolation_proxy_df=pd.DataFrame(sensor_reconstruction_rows)
display(sensor_interpolation_proxy_df)


k=3; true=sensor_state[p_test]; pred=sensor_predictions[k][p_test]
sensor_detail_rows=[]
for j,name in enumerate(sensor_cols):
    neigh=sensor_neighbors[k][j]
    sensor_detail_rows.append({
        "sensor":name,"test_MAE_C":np.mean(np.abs(true[:,j]-pred[:,j])),
        "test_RMSE_C":np.sqrt(np.mean((true[:,j]-pred[:,j])**2)),
        "neighbor_1":sensor_cols[neigh[0]],"neighbor_2":sensor_cols[neigh[1]],"neighbor_3":sensor_cols[neigh[2]],
        "min_neighbor_correlation":np.min(sensor_corr[j,neigh]),
    })
sensor_interpolation_detail_df=pd.DataFrame(sensor_detail_rows).sort_values("test_MAE_C",ascending=False)
display(sensor_interpolation_detail_df.head(30))

sensor_interpolation_proxy_df.to_csv("sensor_interpolation_proxy_summary.csv",index=False)
sensor_interpolation_detail_df.to_csv("sensor_interpolation_proxy_detail.csv",index=False)
print('These neighbors are selected by thermal correlation, not FEM topology.')


In [ ]:
SENSOR_COORDINATE_CANDIDATES=["sensor_coordinates.csv","SRT_sensor_coordinates.csv"]
coordinate_path=None
for directory in DATA_DIRS:
    for filename in SENSOR_COORDINATE_CANDIDATES:
        candidate=directory/filename
        if candidate.exists(): coordinate_path=candidate; break
    if coordinate_path is not None: break

if coordinate_path is None:
    print('Sensor coordinates are unavailable; cannot compare topology-3, IDW, RBF and kriging directly.')
    print('Required format: sensor,x,y,z, with one row for each of the 295 sensors.')
else:
    coords=pd.read_csv(coordinate_path)
    required={"sensor","x","y","z"}
    if not required.issubset(coords.columns):
        raise ValueError(f"The coordinate file must contain {sorted(required)}")
    print('Coordinates loaded:',coordinate_path,'| rows:',len(coords))
    display(coords.head())
    print('The geometric IDW/RBF comparison requires verified sensor-to-FEM-node ordering.')


## 7. Prova di emulatore FEM per inferenza veloce

Questa è una prova di fattibilità, non il surrogato definitivo. L'emulatore viene addestrato sulle coppie disponibili per riprodurre l'output FEM a partire dalle feature termiche. In produzione dovrebbe essere addestrato sul grande dataset sintetico FEM e validato separatamente.


In [ ]:
surrogate_rows=[]
surrogate_predictions={}
for axis in ["XEL","EL"]:
    fem=paired[f"{axis}_FEM_error"].to_numpy(); y_axis=paired[f"{axis}_real_error"].to_numpy()
    surrogate_candidates={
        "ridge":candidate_models["ridge"],
        "random_forest":candidate_models["random_forest"],
    }

    val_scores=[]; fitted={}
    for name,model in surrogate_candidates.items():
        m=clone(model).fit(F_pair.iloc[p_train][thermal_cols],fem[p_train]); fitted[name]=m
        pred=m.predict(F_pair.iloc[p_val][thermal_cols]); val_scores.append((mean_absolute_error(fem[p_val],pred),name))
    best_sur_name=min(val_scores)[1]; surrogate=fitted[best_sur_name]


    residual=y_axis-fem
    correction=clone(candidate_models["ridge"]).fit(F_pair.iloc[p_train][thermal_cols],residual[p_train])
    for split in ["val","test"]:
        idx=split_map[split]
        fem_hat=surrogate.predict(F_pair.iloc[idx][thermal_cols])
        delta_hat=correction.predict(F_pair.iloc[idx][thermal_cols])
        final=fem_hat+delta_hat
        surrogate_rows.append({"surrogate_model":best_sur_name,"FEM_emulator_MAE":mean_absolute_error(fem[idx],fem_hat),**regression_metrics(axis,"surrogate_plus_residual",split,y_axis[idx],final,"no_FEM_online")})
        surrogate_predictions[(axis,split)]=final

surrogate_fem_df=pd.DataFrame(surrogate_rows)
display(surrogate_fem_df)
print('Compare the final result with thermal_only; equivalent performance means the decomposition adds no information.')


## 8. Output per calibrazione fisica del FEM

Questa sezione non modifica coefficienti fisici, perché il CSV contiene soltanto gli offset finali. Esporta invece i casi prioritari da riesaminare nel simulatore. Per una vera inverse calibration servono anche configurazione FEM, carichi, condizioni al contorno e possibilmente deformazioni intermedie.


In [ ]:
physics_review_df=comparison[["Timestamp","SRT Elevation Ange[DEG]","Azimuth","XEL_real_error","XEL_FEM_error","EL_real_error","EL_FEM_error"]].copy()
physics_review_df["XEL_residual"]=physics_review_df.XEL_real_error-physics_review_df.XEL_FEM_error
physics_review_df["EL_residual"]=physics_review_df.EL_real_error-physics_review_df.EL_FEM_error
physics_review_df["joint_residual_norm"]=np.hypot(physics_review_df.XEL_residual,physics_review_df.EL_residual)
physics_review_df["month"]=physics_review_df.Timestamp.dt.month
physics_review_df=physics_review_df.sort_values("joint_residual_norm",ascending=False)
display(physics_review_df.head(30))

alignment_df.to_csv("fem_real_alignment_audit.csv",index=False)
real_fem_audit_df.to_csv("fem_real_global_audit.csv",index=False)
simple_calibration_df.to_csv("fem_real_simple_calibration.csv",index=False)
ablation_df.to_csv("fem_real_multifidelity_ablation.csv",index=False)
selected_ablation_df.to_csv("fem_real_selected_models.csv",index=False)
multifidelity_cv_df.to_csv("fem_real_blocked_cv.csv",index=False)
surrogate_fem_df.to_csv("fem_surrogate_results.csv",index=False)
physics_review_df.to_csv("fem_physics_review_priority.csv",index=False)
print('Export completed.')


## 9. Diagnostica azimutale out-of-fold

Questa sezione verifica se l'azimuth aggiunge informazione predittiva dopo avere controllato temperatura, elevazione e contesto temporale. PCA, scaler e regressori vengono rifittati in ogni fold usando esclusivamente il passato. I risultati sono associativi: senza banda/ricevitore e condizioni meteorologiche non permettono di attribuire causalita' all'azimuth.


In [ ]:
required_previous_objects = [
    'paired', 'X_pair', 'X_elev_pair', 'geometry_fold_bounds', 'FeatureBuilder'
]
missing_previous_objects = [name for name in required_previous_objects if name not in globals()]
if missing_previous_objects:
    raise RuntimeError(
        'Run blocks 0-24 first. Missing objects: ' + ', '.join(missing_previous_objects)
    )
if paired['Azimuth'].isna().any():
    raise ValueError('Azimuth diagnostics require valid Azimuth values for all paired samples.')
if not paired['Timestamp'].is_monotonic_increasing:
    raise ValueError('paired must be chronologically ordered before blocked CV.')
if len(paired) != len(X_pair) or len(paired) != len(X_elev_pair):
    raise ValueError('Metadata and paired thermal windows are misaligned.')

AZ_DIAGNOSTIC_HARMONICS = 3
AZ_OPERATIONAL_ELEVATION = (20.0, 75.0)

def add_azimuth_diagnostic_features(feature_df, meta_df):
    out = feature_df.copy().reset_index(drop=True)
    out = out.drop(columns=['az_sin', 'az_cos'], errors='ignore')
    az = pd.to_numeric(meta_df['Azimuth'], errors='coerce').reset_index(drop=True)
    elev = pd.to_numeric(meta_df['Elevation'], errors='coerce').reset_index(drop=True)
    az_rad = np.deg2rad(az)
    el_centered = (elev - 50.0) / 30.0
    for harmonic in range(1, AZ_DIAGNOSTIC_HARMONICS + 1):
        sin_name, cos_name = f'az_sin_{harmonic}', f'az_cos_{harmonic}'
        out[sin_name] = np.sin(harmonic * az_rad)
        out[cos_name] = np.cos(harmonic * az_rad)
        out[f'{sin_name}_x_el'] = out[sin_name] * el_centered
        out[f'{cos_name}_x_el'] = out[cos_name] * el_centered

    for thermal_name in ['current_std_spatial', 'trend_mean']:
        if thermal_name in out:
            out[f'az_sin_1_x_{thermal_name}'] = out['az_sin_1'] * out[thermal_name]
            out[f'az_cos_1_x_{thermal_name}'] = out['az_cos_1'] * out[thermal_name]
    return out

def azimuth_feature_sets(columns):
    az_all = [c for c in columns if c.startswith('az_')]
    base = [c for c in columns if c not in az_all and c not in ['XEL_FEM_error', 'EL_FEM_error']]
    h1 = [c for c in az_all if c in ['az_sin_1', 'az_cos_1']]
    h3 = [c for c in az_all if ('_x_' not in c)]
    return {
        'without_azimuth': base,
        'azimuth_h1': base + h1,
        'azimuth_h3': base + h3,
        'azimuth_h3_interactions': base + az_all,
    }

az_models = {
    'ridge': Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scale', StandardScaler()),
        ('reg', RidgeCV(alphas=np.logspace(-4, 5, 30))),
    ]),
    'huber': Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scale', RobustScaler()),
        ('reg', HuberRegressor(epsilon=1.5, alpha=1.0, max_iter=4000)),
    ]),
}

azimuth_oof_rows = []
for fold, (train_end, eval_end) in enumerate(geometry_fold_bounds, start=1):
    tr = np.arange(train_end); ev = np.arange(train_end, eval_end)
    # Refit PCA within each fold to exclude future covariates.
    fb_fold = FeatureBuilder(n_pca_current=8, n_pca_trend=8).fit(X_pair[tr], X_elev_pair[tr])
    F_fold = fb_fold.transform(X_pair, X_elev_pair, paired).reset_index(drop=True)
    F_fold = add_azimuth_diagnostic_features(F_fold, paired)
    feature_sets = azimuth_feature_sets(F_fold.columns)
    for axis in ['XEL', 'EL']:
        target = paired[f'{axis}_real_error'].to_numpy(float)
        for feature_set, cols in feature_sets.items():
            for model_name, model in az_models.items():
                fitted = clone(model).fit(F_fold.iloc[tr][cols], target[tr])
                prediction = fitted.predict(F_fold.iloc[ev][cols])
                for pos, pred in zip(ev, prediction):
                    azimuth_oof_rows.append({
                        'axis': axis, 'fold': fold, 'position': int(pos),
                        'Timestamp': paired.iloc[pos]['Timestamp'],
                        'Elevation': paired.iloc[pos]['Elevation'],
                        'Azimuth': paired.iloc[pos]['Azimuth'],
                        'feature_set': feature_set, 'model': model_name,
                        'y_true': target[pos], 'y_pred': pred,
                        'residual': target[pos] - pred,
                        'abs_error': abs(target[pos] - pred),
                    })

azimuth_oof_predictions_df = pd.DataFrame(azimuth_oof_rows)
azimuth_feature_cv_df = azimuth_oof_predictions_df.groupby(
    ['axis', 'model', 'feature_set', 'fold'], as_index=False
).agg(
    count=('abs_error', 'size'), MAE=('abs_error', 'mean'),
    bias=('residual', 'mean'), within_5_percent=('abs_error', lambda x: 100*np.mean(x <= SPEC_ARCSEC)),
    p90_abs_error=('abs_error', lambda x: x.quantile(.90)),
)
azimuth_feature_summary_df = azimuth_feature_cv_df.groupby(
    ['axis', 'model', 'feature_set'], as_index=False
).agg(
    folds=('fold', 'count'), MAE_mean=('MAE', 'mean'), MAE_std=('MAE', 'std'),
    bias_mean=('bias', 'mean'), within_5_mean=('within_5_percent', 'mean'),
    p90_mean=('p90_abs_error', 'mean'),
)
display(azimuth_feature_cv_df)
print('Azimuth ablation across temporal folds')
display(azimuth_feature_summary_df.sort_values(['axis', 'model', 'MAE_mean']))


In [ ]:
def temporal_block_bootstrap_delta(base_df, candidate_df, n_boot=5000, block_days=14, seed=RANDOM_SEED):
    merged = base_df[['position', 'Timestamp', 'abs_error']].merge(
        candidate_df[['position', 'abs_error']], on='position', suffixes=('_base', '_candidate'), validate='one_to_one'
    ).sort_values('Timestamp')
    origin = merged['Timestamp'].min()
    merged['time_block'] = ((merged['Timestamp'] - origin).dt.total_seconds() // (block_days*86400)).astype(int)
    blocks = [g.index.to_numpy() for _, g in merged.groupby('time_block')]
    observed = (merged['abs_error_candidate'] - merged['abs_error_base']).mean()
    rng = np.random.default_rng(seed)
    boot = np.empty(n_boot)
    for i in range(n_boot):
        sampled = rng.integers(0, len(blocks), size=len(blocks))
        idx = np.concatenate([blocks[j] for j in sampled])
        boot[i] = (merged.loc[idx, 'abs_error_candidate'] - merged.loc[idx, 'abs_error_base']).mean()
    return observed, np.quantile(boot, .025), np.quantile(boot, .975), 100*np.mean(boot < 0), len(merged), len(blocks)

azimuth_bootstrap_rows = []
for (axis, model), group in azimuth_oof_predictions_df.groupby(['axis', 'model']):
    base = group[group['feature_set'] == 'without_azimuth']
    for feature_set in ['azimuth_h1', 'azimuth_h3', 'azimuth_h3_interactions']:
        candidate = group[group['feature_set'] == feature_set]
        delta, low, high, probability, count, blocks = temporal_block_bootstrap_delta(base, candidate)
        azimuth_bootstrap_rows.append({
            'axis': axis, 'model': model, 'feature_set': feature_set,
            'delta_MAE_vs_without_azimuth': delta, 'ci95_low': low, 'ci95_high': high,
            'probability_better_percent': probability, 'count': count, 'temporal_blocks': blocks,
        })
azimuth_feature_bootstrap_df = pd.DataFrame(azimuth_bootstrap_rows)
display(azimuth_feature_bootstrap_df.sort_values(['axis', 'model', 'delta_MAE_vs_without_azimuth']))


In [ ]:
AZ_BINS = np.arange(0, 361, 45)
ELEV_DIAGNOSTIC_BINS = [20, 30, 45, 60, 75]


baseline_model_by_axis = {}
for axis in ['XEL', 'EL']:
    candidates = azimuth_feature_summary_df[
        (azimuth_feature_summary_df.axis == axis) &
        (azimuth_feature_summary_df.feature_set == 'without_azimuth')
    ]
    baseline_model_by_axis[axis] = candidates.loc[candidates.MAE_mean.idxmin(), 'model']
print('Diagnostic model without azimuth:', baseline_model_by_axis)

diagnostic_parts = []
for axis, model in baseline_model_by_axis.items():
    part = azimuth_oof_predictions_df[
        (azimuth_oof_predictions_df.axis == axis) &
        (azimuth_oof_predictions_df.model == model) &
        (azimuth_oof_predictions_df.feature_set == 'without_azimuth')
    ].copy()
    part['az_band'] = pd.cut(part.Azimuth, AZ_BINS, include_lowest=True, right=False)
    part['elev_band'] = pd.cut(part.Elevation, ELEV_DIAGNOSTIC_BINS, include_lowest=True)
    part['month'] = part.Timestamp.dt.month
    part['season'] = ((part['month'] % 12) // 3).astype(int)
    diagnostic_parts.append(part)
azimuth_oof_diagnostic_df = pd.concat(diagnostic_parts, ignore_index=True)
operational = azimuth_oof_diagnostic_df[
    azimuth_oof_diagnostic_df.Elevation.between(*AZ_OPERATIONAL_ELEVATION)
].copy()
azimuth_residual_by_sector_df = operational.groupby(
    ['axis', 'az_band'], observed=True, as_index=False
).agg(
    count=('abs_error', 'size'), bias=('residual', 'mean'), MAE=('abs_error', 'mean'),
    within_5_percent=('abs_error', lambda x: 100*np.mean(x <= SPEC_ARCSEC)),
    p90_abs_error=('abs_error', lambda x: x.quantile(.90) if len(x) else np.nan),
    extreme_error_percent=('abs_error', lambda x: 100*np.mean(x > 15)),
)
print('Out-of-fold residuals by sector, elevation 20-75 deg')
display(azimuth_residual_by_sector_df)

def sector_statistic(values, sectors):
    frame = pd.DataFrame({'value': values, 'sector': sectors}).dropna()
    grouped = frame.groupby('sector').value.agg(['mean', 'count'])
    grouped = grouped[grouped['count'] >= 10]
    if len(grouped) < 2:
        return np.nan
    grand = np.average(grouped['mean'], weights=grouped['count'])
    return np.average((grouped['mean'] - grand)**2, weights=grouped['count'])

def stratified_sector_permutation(frame, value_col, n_perm=5000, seed=RANDOM_SEED):
    d = frame.dropna(subset=[value_col, 'az_band', 'elev_band', 'season']).copy()
    sectors = d['az_band'].astype(str).to_numpy()
    values = d[value_col].to_numpy(float)
    strata = (d['elev_band'].astype(str) + '_s' + d['season'].astype(str)).to_numpy()
    observed = sector_statistic(values, sectors)
    rng = np.random.default_rng(seed)
    exceed = 0
    permuted = sectors.copy()
    stratum_indices = [np.flatnonzero(strata == s) for s in np.unique(strata)]
    for _ in range(n_perm):
        for idx in stratum_indices:
            permuted[idx] = rng.permutation(sectors[idx])
        stat = sector_statistic(values, permuted)
        exceed += int(np.isfinite(stat) and stat >= observed)
    return observed, (exceed + 1) / (n_perm + 1), len(d)

conditional_rows = []
for axis, part in operational.groupby('axis'):
    for value_col in ['residual', 'abs_error']:
        statistic, p_value, count = stratified_sector_permutation(part, value_col)
        conditional_rows.append({
            'axis': axis, 'quantity': value_col, 'count': count,
            'sector_between_variance': statistic, 'stratified_permutation_p': p_value,
            'controlled_for': 'elevation_band+season',
        })
azimuth_conditional_test_df = pd.DataFrame(conditional_rows)
print('Azimuth test conditional on elevation band and season')
display(azimuth_conditional_test_df)

fig, axs = plt.subplots(2, 2, figsize=(15, 10))
for row, axis in enumerate(['XEL', 'EL']):
    table = operational[operational.axis == axis].pivot_table(
        index='elev_band', columns='az_band', values='abs_error', aggfunc='mean', observed=True
    )
    counts = operational[operational.axis == axis].pivot_table(
        index='elev_band', columns='az_band', values='abs_error', aggfunc='size', observed=True
    )
    masked = table.mask(counts < 5)
    image = axs[row, 0].imshow(masked, aspect='auto', cmap='magma', vmin=0)
    axs[row, 0].set(title=f'{axis}: MAE OOF [arcsec]', xlabel='Azimuth sector', ylabel='Elevation band')
    axs[row, 0].set_xticks(range(len(masked.columns)), [str(x) for x in masked.columns], rotation=45, ha='right')
    axs[row, 0].set_yticks(range(len(masked.index)), [str(x) for x in masked.index])
    for i in range(masked.shape[0]):
        for j in range(masked.shape[1]):
            if np.isfinite(masked.iloc[i, j]): axs[row, 0].text(j, i, f'{masked.iloc[i,j]:.1f}\nn={int(counts.iloc[i,j])}', ha='center', va='center', color='white', fontsize=8)
    plt.colorbar(image, ax=axs[row, 0], label='MAE [arcsec]')
    sector = azimuth_residual_by_sector_df[azimuth_residual_by_sector_df.axis == axis]
    x = np.arange(len(sector))
    axs[row, 1].bar(x, sector['bias'], alpha=.8, label='signed bias')
    axs[row, 1].plot(x, sector['MAE'], 'ko-', label='MAE')
    axs[row, 1].axhline(0, color='k', lw=1)
    axs[row, 1].set_xticks(x, [str(v) for v in sector.az_band], rotation=45, ha='right')
    axs[row, 1].set(title=f'{axis}: residual by azimuth (EL 20-75 deg)', ylabel='Arcsec')
    axs[row, 1].grid(True, axis='y', alpha=.25); axs[row, 1].legend()
plt.tight_layout(); plt.show()


In [ ]:
decision_rows = []
for axis in ['XEL', 'EL']:
    candidates = azimuth_feature_bootstrap_df[azimuth_feature_bootstrap_df.axis == axis].copy()
    best = candidates.loc[candidates.delta_MAE_vs_without_azimuth.idxmin()]
    conditional = azimuth_conditional_test_df[
        (azimuth_conditional_test_df.axis == axis) &
        (azimuth_conditional_test_df.quantity == 'abs_error')
    ].iloc[0]
    decision_rows.append({
        'axis': axis, 'best_azimuth_variant': best.feature_set, 'model': best.model,
        'delta_MAE': best.delta_MAE_vs_without_azimuth,
        'ci95_low': best.ci95_low, 'ci95_high': best.ci95_high,
        'probability_better_percent': best.probability_better_percent,
        'conditional_abs_error_p': conditional.stratified_permutation_p,
        'predictive_gain_supported': bool(best.ci95_high < 0),
        'sector_association_supported': bool(conditional.stratified_permutation_p < .05),
    })
azimuth_decision_df = pd.DataFrame(decision_rows)
display(azimuth_decision_df)

azimuth_oof_predictions_df.to_csv('azimuth_oof_predictions.csv', index=False)
azimuth_feature_cv_df.to_csv('azimuth_feature_blocked_cv.csv', index=False)
azimuth_feature_summary_df.to_csv('azimuth_feature_summary.csv', index=False)
azimuth_feature_bootstrap_df.to_csv('azimuth_feature_temporal_bootstrap.csv', index=False)
azimuth_residual_by_sector_df.to_csv('azimuth_oof_residual_by_sector.csv', index=False)
azimuth_conditional_test_df.to_csv('azimuth_conditional_permutation.csv', index=False)
azimuth_decision_df.to_csv('azimuth_decision_summary.csv', index=False)
print('Azimuth-diagnostic export completed.')


## 10. Geometria solare e orientamento relativo

I timestamp sono UTC. La posizione solare viene calcolata alle coordinate SRT 39°29'32.2\"N, 9°14'46.8\"E. Le feature descrivono soltanto la geometria Sole-telescopio: non misurano nuvolosita', radiazione effettiva o ombre locali. Il confronto resta out-of-fold e usa gli stessi blocchi temporali della diagnostica azimutale.


In [ ]:
SRT_LATITUDE_DEG = 39 + 29/60 + 32.2/3600
SRT_LONGITUDE_DEG = 9 + 14/60 + 46.8/3600

def solar_position_utc(timestamps, latitude_deg=SRT_LATITUDE_DEG, longitude_deg=SRT_LONGITUDE_DEG):
    # NOAA solar equations provide geometric features, not measured irradiance.
    ts = pd.to_datetime(pd.Series(timestamps), utc=True)
    unix_seconds = ts.astype('int64').to_numpy(dtype=float) / 1e9
    julian_day = unix_seconds / 86400.0 + 2440587.5
    century = (julian_day - 2451545.0) / 36525.0
    mean_long = np.mod(280.46646 + century*(36000.76983 + century*0.0003032), 360.0)
    mean_anomaly = 357.52911 + century*(35999.05029 - 0.0001537*century)
    eccentricity = 0.016708634 - century*(0.000042037 + 0.0000001267*century)
    anomaly_rad = np.deg2rad(mean_anomaly)
    center = (
        np.sin(anomaly_rad)*(1.914602 - century*(0.004817 + 0.000014*century))
        + np.sin(2*anomaly_rad)*(0.019993 - 0.000101*century)
        + np.sin(3*anomaly_rad)*0.000289
    )
    true_long = mean_long + center
    omega = 125.04 - 1934.136*century
    apparent_long = true_long - 0.00569 - 0.00478*np.sin(np.deg2rad(omega))
    mean_obliquity = 23 + (26 + (21.448 - century*(46.815 + century*(0.00059 - century*0.001813)))/60)/60
    obliquity = mean_obliquity + 0.00256*np.cos(np.deg2rad(omega))
    obliquity_rad = np.deg2rad(obliquity)
    declination = np.arcsin(np.sin(obliquity_rad)*np.sin(np.deg2rad(apparent_long)))
    y_term = np.tan(obliquity_rad/2)**2
    mean_long_rad = np.deg2rad(mean_long)
    equation_time = 4*np.rad2deg(
        y_term*np.sin(2*mean_long_rad) - 2*eccentricity*np.sin(anomaly_rad)
        + 4*eccentricity*y_term*np.sin(anomaly_rad)*np.cos(2*mean_long_rad)
        - 0.5*y_term**2*np.sin(4*mean_long_rad)
        - 1.25*eccentricity**2*np.sin(2*anomaly_rad)
    )
    utc_minutes = (
        ts.dt.hour.to_numpy()*60 + ts.dt.minute.to_numpy()
        + ts.dt.second.to_numpy()/60 + ts.dt.microsecond.to_numpy()/6e7
    )
    true_solar_minutes = np.mod(utc_minutes + equation_time + 4*longitude_deg, 1440.0)
    hour_angle_deg = true_solar_minutes/4 - 180
    hour_angle = np.deg2rad(hour_angle_deg)
    latitude = np.deg2rad(latitude_deg)
    cos_zenith = (
        np.sin(latitude)*np.sin(declination)
        + np.cos(latitude)*np.cos(declination)*np.cos(hour_angle)
    )
    cos_zenith = np.clip(cos_zenith, -1, 1)
    elevation = 90 - np.rad2deg(np.arccos(cos_zenith))
    azimuth = np.mod(np.rad2deg(np.arctan2(
        np.sin(hour_angle),
        np.cos(hour_angle)*np.sin(latitude) - np.tan(declination)*np.cos(latitude),
    )) + 180, 360)
    return pd.DataFrame({'solar_elevation': elevation, 'solar_azimuth': azimuth})

def solar_geometry_features(meta_df):
    out = solar_position_utc(meta_df['Timestamp']).reset_index(drop=True)
    telescope_az = pd.to_numeric(meta_df['Azimuth'], errors='coerce').to_numpy(float)
    telescope_el = pd.to_numeric(meta_df['Elevation'], errors='coerce').to_numpy(float)
    relative_az = (telescope_az - out['solar_azimuth'].to_numpy() + 180) % 360 - 180
    sun_el_rad = np.deg2rad(out['solar_elevation'].to_numpy())
    sun_az_rad = np.deg2rad(out['solar_azimuth'].to_numpy())
    tel_el_rad = np.deg2rad(telescope_el)
    rel_rad = np.deg2rad(relative_az)
    daylight_proxy = np.maximum(np.sin(sun_el_rad), 0.0)
    separation_cos = (
        np.sin(tel_el_rad)*np.sin(sun_el_rad)
        + np.cos(tel_el_rad)*np.cos(sun_el_rad)*np.cos(rel_rad)
    )
    out['sun_up'] = (out['solar_elevation'] > 0).astype(float)
    out['solar_height_proxy'] = daylight_proxy
    out['solar_az_sin'] = np.sin(sun_az_rad)
    out['solar_az_cos'] = np.cos(sun_az_rad)
    out['relative_sun_azimuth'] = relative_az
    out['relative_sun_sin'] = np.sin(rel_rad)
    out['relative_sun_cos'] = np.cos(rel_rad)
    out['irradiated_side_sin_proxy'] = daylight_proxy*np.sin(rel_rad)
    out['irradiated_side_cos_proxy'] = daylight_proxy*np.cos(rel_rad)
    out['sun_pointing_separation_cos'] = separation_cos
    # Geometric illumination history does not account for clouds.
    for lag_hours in [1, 2, 4]:
        lagged = solar_position_utc(pd.to_datetime(meta_df['Timestamp']) - pd.Timedelta(hours=lag_hours))
        out[f'solar_height_proxy_lag{lag_hours}h'] = np.maximum(
            np.sin(np.deg2rad(lagged['solar_elevation'].to_numpy())), 0.0
        )
    return out

solar_pair = solar_geometry_features(paired)
solar_audit_df = pd.DataFrame([{
    'samples': len(solar_pair),
    'sun_up_percent': 100*solar_pair.sun_up.mean(),
    'solar_elevation_min': solar_pair.solar_elevation.min(),
    'solar_elevation_max': solar_pair.solar_elevation.max(),
    'solar_azimuth_min': solar_pair.solar_azimuth.min(),
    'solar_azimuth_max': solar_pair.solar_azimuth.max(),
}])
display(solar_audit_df)
display(pd.concat([paired[['Timestamp', 'Elevation', 'Azimuth']].head(), solar_pair.head()], axis=1))


In [ ]:
solar_oof_rows = []
solar_absolute_cols = [
    'solar_elevation', 'solar_az_sin', 'solar_az_cos', 'sun_up', 'solar_height_proxy',
    'solar_height_proxy_lag1h', 'solar_height_proxy_lag2h', 'solar_height_proxy_lag4h',
]
solar_relative_cols = [
    'relative_sun_sin', 'relative_sun_cos', 'irradiated_side_sin_proxy',
    'irradiated_side_cos_proxy', 'sun_pointing_separation_cos',
]
for fold, (train_end, eval_end) in enumerate(geometry_fold_bounds, start=1):
    tr = np.arange(train_end); ev = np.arange(train_end, eval_end)
    fb_fold = FeatureBuilder(n_pca_current=8, n_pca_trend=8).fit(X_pair[tr], X_elev_pair[tr])
    features = fb_fold.transform(X_pair, X_elev_pair, paired).reset_index(drop=True)
    features = features.drop(columns=['az_sin', 'az_cos'], errors='ignore')
    features = pd.concat([features, solar_pair.reset_index(drop=True)], axis=1)
    solar_all = solar_absolute_cols + solar_relative_cols
    base_cols = [c for c in features.columns if c not in solar_all and c not in ['solar_azimuth', 'relative_sun_azimuth']]
    feature_sets = {
        'thermal_time_without_sun': base_cols,
        'thermal_plus_solar_absolute': base_cols + solar_absolute_cols,
        'thermal_plus_relative_sun': base_cols + solar_relative_cols,
        'thermal_plus_all_solar': base_cols + solar_all,
    }
    for axis in ['XEL', 'EL']:
        target = paired[f'{axis}_real_error'].to_numpy(float)
        for model_name, model in az_models.items():
            for feature_set, cols in feature_sets.items():
                fitted = clone(model).fit(features.iloc[tr][cols], target[tr])
                prediction = fitted.predict(features.iloc[ev][cols])
                for pos, pred in zip(ev, prediction):
                    solar_oof_rows.append({
                        'axis': axis, 'model': model_name, 'feature_set': feature_set,
                        'fold': fold, 'position': int(pos), 'Timestamp': paired.iloc[pos].Timestamp,
                        'sun_up': solar_pair.iloc[pos].sun_up,
                        'solar_elevation': solar_pair.iloc[pos].solar_elevation,
                        'relative_sun_azimuth': solar_pair.iloc[pos].relative_sun_azimuth,
                        'y_true': target[pos], 'y_pred': pred,
                        'residual': target[pos]-pred, 'abs_error': abs(target[pos]-pred),
                    })
solar_oof_predictions_df = pd.DataFrame(solar_oof_rows)
solar_feature_cv_df = solar_oof_predictions_df.groupby(
    ['axis', 'model', 'feature_set', 'fold'], as_index=False
).agg(
    count=('abs_error', 'size'), MAE=('abs_error', 'mean'), bias=('residual', 'mean'),
    within_5_percent=('abs_error', lambda x: 100*np.mean(x <= SPEC_ARCSEC)),
    p90_abs_error=('abs_error', lambda x: x.quantile(.90)),
)
solar_feature_summary_df = solar_feature_cv_df.groupby(
    ['axis', 'model', 'feature_set'], as_index=False
).agg(
    folds=('fold', 'count'), MAE_mean=('MAE', 'mean'), MAE_std=('MAE', 'std'),
    bias_mean=('bias', 'mean'), within_5_mean=('within_5_percent', 'mean'),
    p90_mean=('p90_abs_error', 'mean'),
)
display(solar_feature_summary_df.sort_values(['axis', 'model', 'MAE_mean']))


In [ ]:
solar_bootstrap_rows = []
for (axis, model), group in solar_oof_predictions_df.groupby(['axis', 'model']):
    base = group[group.feature_set == 'thermal_time_without_sun']
    for feature_set in [
        'thermal_plus_solar_absolute', 'thermal_plus_relative_sun', 'thermal_plus_all_solar'
    ]:
        candidate = group[group.feature_set == feature_set]
        delta, low, high, probability, count, blocks = temporal_block_bootstrap_delta(base, candidate)
        solar_bootstrap_rows.append({
            'axis': axis, 'model': model, 'feature_set': feature_set,
            'delta_MAE_vs_without_sun': delta, 'ci95_low': low, 'ci95_high': high,
            'probability_better_percent': probability, 'count': count, 'temporal_blocks': blocks,
        })
solar_feature_bootstrap_df = pd.DataFrame(solar_bootstrap_rows)
display(solar_feature_bootstrap_df.sort_values(['axis', 'model', 'delta_MAE_vs_without_sun']))

solar_daynight_df = solar_oof_predictions_df.groupby(
    ['axis', 'model', 'feature_set', 'sun_up'], as_index=False
).agg(
    count=('abs_error', 'size'), MAE=('abs_error', 'mean'), bias=('residual', 'mean'),
    within_5_percent=('abs_error', lambda x: 100*np.mean(x <= SPEC_ARCSEC)),
    p90_abs_error=('abs_error', lambda x: x.quantile(.90)),
)
print('Out-of-fold performance: night versus day')
display(solar_daynight_df)

fig, axs = plt.subplots(1, 2, figsize=(14, 5))
for ax, axis in zip(axs, ['XEL', 'EL']):
    model = baseline_model_by_axis[axis]
    plot_data = solar_oof_predictions_df[
        (solar_oof_predictions_df.axis == axis) &
        (solar_oof_predictions_df.model == model) &
        (solar_oof_predictions_df.feature_set == 'thermal_time_without_sun')
    ]
    sc = ax.scatter(
        plot_data.relative_sun_azimuth, plot_data.abs_error,
        c=plot_data.solar_elevation, cmap='coolwarm', alpha=.75, s=28,
    )
    ax.axhline(SPEC_ARCSEC, color='green', ls='--', lw=1)
    ax.set(title=f'{axis}: error versus orientation relative to the Sun',
           xlabel='Telescope azimuth - solar azimuth [deg]', ylabel='OOF absolute error [arcsec]')
    ax.grid(True, alpha=.25); plt.colorbar(sc, ax=ax, label='Solar elevation [deg]')
plt.tight_layout(); plt.show()


In [ ]:
solar_decision_rows = []
for axis in ['XEL', 'EL']:
    selected_model = baseline_model_by_axis[axis]
    candidates = solar_feature_bootstrap_df[
        (solar_feature_bootstrap_df.axis == axis) &
        (solar_feature_bootstrap_df.model == selected_model)
    ]
    best = candidates.loc[candidates.delta_MAE_vs_without_sun.idxmin()]
    baseline_summary = solar_feature_summary_df[
        (solar_feature_summary_df.axis == axis) &
        (solar_feature_summary_df.model == selected_model) &
        (solar_feature_summary_df.feature_set == 'thermal_time_without_sun')
    ].iloc[0]
    candidate_summary = solar_feature_summary_df[
        (solar_feature_summary_df.axis == axis) &
        (solar_feature_summary_df.model == selected_model) &
        (solar_feature_summary_df.feature_set == best.feature_set)
    ].iloc[0]
    gain_supported = bool(best.ci95_high < 0)
    solar_decision_rows.append({
        'axis': axis, 'selected_model': selected_model,
        'baseline_feature_set': 'thermal_time_without_sun',
        'baseline_MAE_mean': baseline_summary.MAE_mean,
        'best_solar_variant': best.feature_set,
        'solar_variant_MAE_mean': candidate_summary.MAE_mean,
        'delta_MAE': best.delta_MAE_vs_without_sun,
        'ci95_low': best.ci95_low, 'ci95_high': best.ci95_high,
        'probability_better_percent': best.probability_better_percent,
        'predictive_gain_supported': gain_supported,
        'recommended_feature_set': best.feature_set if gain_supported else 'thermal_time_without_sun',
    })
solar_decision_df = pd.DataFrame(solar_decision_rows)
display(solar_decision_df)

solar_pair.to_csv('solar_geometry_for_clean_windows.csv', index=False)
solar_oof_predictions_df.to_csv('solar_oof_predictions.csv', index=False)
solar_feature_cv_df.to_csv('solar_feature_blocked_cv.csv', index=False)
solar_feature_summary_df.to_csv('solar_feature_summary.csv', index=False)
solar_feature_bootstrap_df.to_csv('solar_feature_temporal_bootstrap.csv', index=False)
solar_daynight_df.to_csv('solar_daynight_performance.csv', index=False)
solar_decision_df.to_csv('solar_decision_summary.csv', index=False)
print('Solar-diagnostic export completed.')


## Interpretazione decisionale

- Se `affine_FEM` migliora stabilmente: calibrazione semplice di bias/scala.
- Se `thermal_plus_FEM` supera `thermal_only`: il FEM aggiunge informazione utile.
- Se `FEM_plus_residual` è migliore e stabile: usare un discrepancy model.
- Se il surrogato mantiene il vantaggio: architettura multi-fidelity utilizzabile in tempo reale.
- Se nessuna variante con FEM supera `thermal_only`: intervenire sul simulatore e sulle condizioni al contorno prima di generare altri sintetici.
